# Which local Kubernetes can run GPU work on WSL2

I ran every local Kubernetes I could install on this machine against the same GPU probe, one at a time, resetting the machine to a blank slate between them.

| Step | Question | Answer |
| --- | --- | --- |
| 00 | What does the machine look like with nothing installed? | the reference every step is diffed against |
| 01 | How small a VRAM difference can I see? | 1 to 39 MiB of noise by session; a known 2048 MiB reads as 2048 to 2050 |
| 02 | Does the GPU survive container nesting? | `--gpus` does not nest; three mounts do, at every depth tested |
| 03 | k3s in WSL2 | level 50 stock; rationed with a one-line plugin patch |
| 04 | k3d | nothing stock; level 50 with two node mounts and privileged pods |
| 05 | kind | the same as k3d, through `extraMounts` |
| 06 | minikube | level 10 stock; level 50 with one extra mount |
| 07 | Rancher Desktop | level 50 with hostPath; starts Kubernetes 1.25 by default |
| 08 | Docker Desktop, kind mode | only by copying the driver into every pod |
| 08b | Docker Desktop, kubeadm mode | level 50 with hostPath; RuntimeClass rejected |
| 09 | Which one, and why? | k3s: fastest, lightest, unprivileged pods |
| 10 | Rebuild it | k3s + device plugin + admission policy + keepalive |

Data: [`data/cluster-matrix/`](../../data/cluster-matrix/README.md). Scripts: [`bench/cluster-matrix/`](../../bench/cluster-matrix/README.md).

## Reading the commands

The cells are a record, not a script. They ran from the repo root in Git Bash with `MSYS_NO_PATHCONV=1`, so Linux paths reach `wsl.exe` unchanged.

- `T` is the repo root as WSL sees it. Anything longer than one command went in as a script (`wsl -d Ubuntu -u root -- bash $T/bench/cluster-matrix/tools/<script>`), because `wsl.exe -- <cmd>` expands `$vars` on the Windows side.
- Each cluster wrote its kubeconfig to its step's `raw/` folder, gitignored, so `~/.kube` stayed empty.
- Probe pods went through `bench/cluster-matrix/tools/kpod.sh`, against whatever `KUBECONFIG` points at.
- `bench/cluster-matrix/slate/` (machine listings) and `bench/cluster-matrix/bin/` (a candidate's binary) are gitignored.

Output is sometimes trimmed; the full output is in the step's `raw/` folder.

Since step 10, NeuroMesh runs `gpu-lab`, not these scripts; see [dev-machine.md](../dev-machine.md#the-cluster).

In [ ]:
# Git Bash, from the repo root
export MSYS_NO_PATHCONV=1
T=/mnt/c/PROJECTS/infinity-bench   # the repo root, as WSL sees it

## The rules, fixed before the first result

**A candidate works only at level 50.** The probe reports the highest level it reached:

| Level | Meaning |
| --- | --- |
| 10 | nothing |
| 20 | NVML initialised, so the driver is reachable |
| 30 | CUDA context created |
| 40 | a 2 GiB allocation succeeded |
| **50** | **a GEMM ran and I checked every element of the result** |

`nvidia-smi` printing a device name is only level 20, and `Running` is never evidence: a pod can sit at `Running`, holding `nvidia.com/gpu: 1`, with no driver in it.

**Separately: does `nvidia.com/gpu` become allocatable?** A GPU that only works around the scheduler is not one Kubernetes can ration.

- One probe image, built once and loaded into every candidate by image ID. It must also pass on the bare host in the same session, or the session counts for nothing.
- `NVIDIA_DRIVER_CAPABILITIES=compute,utility` is set explicitly everywhere.
- Stock and tuned are separate results. Tuning gets 45 minutes per candidate, written up whether it works or not.
- A re-run that disagrees is kept, and the candidate is marked flaky.
- Error text goes in whole.
- VRAM is a yes or no against the measured noise floor, never a number below it.
- Every candidate is removed after its step, and the machine is checked against the blank slate before the next one.

## 00. The baseline

**Question:** what does this machine look like with no Kubernetes on it?

**Answer:** the state below. Every candidate starts from it and returns to it.

Run on 2026-10-29.

In [ ]:
bash bench/cluster-matrix/tools/slate-check.sh > bench/cluster-matrix/slate/baseline.txt

The script checks, in order:

| Section | Commands | Where |
| --- | --- | --- |
| WSL distros | `wsl -l -v` | Windows |
| k3s and Kubernetes leftovers | `ls -d /usr/local/bin/k3s* /etc/rancher /var/lib/rancher /var/lib/kubelet ...`, `pgrep -lf 'k3s\|kubelet\|containerd'`, `mount`, `ip -br link`, `iptables-save` | WSL, as root (`bench/cluster-matrix/tools/slate-wsl.sh`) |
| Container toolkit | `dpkg-query -W nvidia-container-toolkit` | WSL, as root |
| Docker | `docker ps -a`, `docker network ls`, `docker volume ls`, `docker images` filtered to cluster images | Docker Desktop |
| Docker Desktop Kubernetes | `KubernetesEnabled` in `%APPDATA%\Docker\settings-store.json` | Windows |
| kubeconfig | `kubectl config get-contexts -o name` | Windows |
| Tools | `command -v kind k3d minikube rdctl`; `minikube profile list` only if `~/.minikube` exists | Windows |
| Leftovers | `ls -d %LOCALAPPDATA%\rancher-desktop ~/.rd ~/.kind ~/.k3d` | Windows |
| GPU | `nvidia-smi.exe --query-gpu=memory.used,memory.total,utilization.gpu` | Windows, not compared |

Any minikube command recreates an empty `~/.minikube`, hence the guard.

### The baseline

| Check | State |
| --- | --- |
| WSL distros | `Ubuntu`, `docker-desktop` |
| Kubernetes in Ubuntu | none: no binaries, unit, processes, mounts, CNI interfaces or iptables rules |
| `nvidia-container-toolkit` | 1.20.1-1 in Ubuntu, kept because k3s needs it |
| Docker Desktop Kubernetes | off |
| Cluster images in Docker | none |
| kind, k3d, rdctl | not installed |
| minikube | installed, no `~/.minikube` |
| Rancher Desktop | not installed |
| kubeconfig | none |
| Runner fleet | 11 containers, stopped for the whole run |
| GPU | 2366 MiB used of 8151, 0 % utilisation, nothing running on it |

Docker volumes and networks from other projects are part of the baseline. Full output: `bench/cluster-matrix/slate/baseline.txt`.

## 01. The instrument

**Question:** every later VRAM claim is a whole-card before and after. How much noise is there, and does a known 2 GiB read as 2 GiB?

**Answer:** the floor is 1 to 39 MiB depending on the session, and a known 2048 MiB reads as 2048 to 2050.

| Setup | |
| --- | --- |
| Run | 2026-10-29 |
| Driver | 621.01, CUDA 13.5 |
| Probe image | `cluster-matrix-probe:01`, `sha256:2ef8cd3ee9a4a67dcf812f1793a2f99bfbe8b8cc1ee8291e73dac2d718574f26` |
| Data | [`01-instrument/`](../../data/cluster-matrix/2026-10-29/01-instrument/) |

### 1. The slate matched the baseline

In [ ]:
bash bench/cluster-matrix/tools/slate-check.sh > bench/cluster-matrix/slate/01-instrument-before.txt
diff bench/cluster-matrix/slate/baseline.txt bench/cluster-matrix/slate/01-instrument-before.txt

< disk used /: 8645M
> disk used /: 8650M
< 2366 MiB, 8151 MiB, 0 %
> 2262 MiB, 8151 MiB, 0 %


Only disk and GPU memory moved; the baseline allows both.

### 2. With the desktop quiet, the floor is 1 MiB

In [ ]:
wsl -d Ubuntu -- bash -c "cd $T && OUT_DIR=$T/data/cluster-matrix/2026-10-29/01-instrument/raw SAMPLES=30 INTERVAL=2 bash bench/cluster-matrix/probe/floor.sh live"

sampling whole-card memory.used: 30 samples, 2s apart, label=live
median 2265 MiB  IQR 0 MiB  range 2265-2266 MiB  spread 1 MiB
floor recorded. a VRAM delta below 1 MiB is not measurable on this machine.


`raw/noise-floor.csv`:

```text
run_id,label,samples,interval_s,median_mib,iqr_mib,min_mib,max_mib,spread_mib,conditions
20261029T160811Z,live,30,2,2265,0,2265,2266,1,"49.80,2535MHz,23.57W,0%,0x0000000000000400"
```

**Finding.** On 2026-10-01 the same minute spread 17 MiB, and a busier 10 seconds spread 31. Today it spread 1 MiB with about 780 MiB less in use, because the runner fleet and Docker Desktop's VM were both stopped (finding 7). The floor belongs to the session, so I measure it again for every candidate.
It also hides slow drift: over the next ten minutes idle moved from 2265 to 2281 MiB (finding 6). Only a difference with its own before and after reading counts.

### 3. The probe had never been compiled

In [ ]:
cd bench/cluster-matrix/probe && docker build -t cluster-matrix-probe:01 .

#9 0.542 /tmp/gpu_probe.cu(85): error: identifier "nvmlDeviceGetComputeRunningProcesses_v2" is undefined
#9 0.546 1 error detected in the compilation of "/tmp/gpu_probe.cu".
ERROR: failed to build: failed to solve: process "/bin/sh -c nvcc -O2 -o /tmp/gpu_probe /tmp/gpu_probe.cu ..." did not complete successfully: exit code: 2


Full log in `errors/build-1-nvml-v2.log`. The build ran 16:08:20 to 16:11:06Z, mostly pulling the CUDA 12.8 devel image.

**Finding.** In CUDA 12.8, `nvml.h` no longer declares the `_v2` call; the unversioned name maps to `_v3`, so I switched to it. It's the per-process query, which returns nothing on WSL2 anyway, which is probably why the break went unnoticed.

### 4. Three changes to the probe

All in `bench/cluster-matrix/probe/gpu_probe.cu`:

- **Hold.** A second argument, `hold_s`, keeps the allocation and context alive after the result prints. Without it the probe exits in milliseconds and a 1-second sampler never sees it.
- **Resident.** `cudaMemset` writes the whole 2 GiB, not just reserves it.
- **Escaping.** JSON strings go through `json_str`, so a quote or backslash in an error can't break the output.

`bench/cluster-matrix/probe/lib.sh` now lets `OUT_DIR` be overridden, so each step writes into its own data folder.

In [ ]:
docker build -t cluster-matrix-probe:01 .
docker image inspect cluster-matrix-probe:01 --format 'id={{.Id}} size={{.Size}}'

exit=0
id=sha256:2ef8cd3ee9a4a67dcf812f1793a2f99bfbe8b8cc1ee8291e73dac2d718574f26 size=98845666


### 5. The bare-host control passes at level 50

L0: the binary from the image, run directly in WSL2.

In [ ]:
cid=$(docker create cluster-matrix-probe:01)
docker cp $cid:/usr/local/bin/gpu_probe bench/cluster-matrix/probe/bin/gpu_probe
docker rm $cid
sha256sum bench/cluster-matrix/probe/bin/gpu_probe

043a17f35d3f2e594f83597051314df3f821c1608dec9a380bd548a3360b9f08 *bench/cluster-matrix/probe/bin/gpu_probe


My first try ran it inline:

In [ ]:
wsl -d Ubuntu -- bash -c 'B=/mnt/c/.../gpu_probe; ldd $B; $B 2048 0'

Try `ldd --help' for more information.
bash: line 1: 2048: command not found


`$B` was expanded on the Windows side (the Git Bash gotcha from gpu-debug.md), so it went into a script, `bench/cluster-matrix/tools/l0-once.sh`:

In [ ]:
wsl -d Ubuntu -- bash $T/bench/cluster-matrix/tools/l0-once.sh

## ldd
	libnvidia-ml.so.1 => /usr/lib/wsl/lib/libnvidia-ml.so.1
	libc.so.6, libpthread.so.0, libm.so.6, libdl.so.2, librt.so.1 => /lib/x86_64-linux-gnu/...
## run: gpu_probe 2048 0
{
  "nvml_device_count": 1,
  "nvml_name": "NVIDIA GeForce RTX 5060",
  "nvml_total_mib": 8151,
  "nvml_compute_procs": 0,
  "cuda_name": "NVIDIA GeForce RTX 5060",
  "compute_capability": "12.0",
  "allocated_mib": 2048,
  "gemm_value": 512.0,
  "hold_s": 0,
  "level": 50,
  "failed_stage": "",
  "error": ""
}
exit=50


**Finding.** Only NVML is linked dynamically, from `/usr/lib/wsl/lib`. The CUDA runtime is static and `libcuda` is opened at run time, so `ldd` doesn't show it. Those two libraries are what each container boundary has to deliver.
`nvml_compute_procs` is 0 even with the probe's own context open: per-process accounting is empty on WSL2.

### 6. A known 2048 MiB allocation reads as 2048 MiB

`bench/cluster-matrix/tools/instrument.sh` runs the probe at 0 MiB (context only) and at 2048 MiB, each held 30 s, and takes the median of 10 one-second samples before, during and after.

The first run stopped silently after one probe:

In [ ]:
wsl -d Ubuntu -- bash $T/bench/cluster-matrix/tools/instrument.sh $T/bench/cluster-matrix/probe/bin/gpu_probe $T/data/cluster-matrix/2026-10-29/01-instrument/raw 3 2048 30

(no output; only raw/probe-a0-t1.json written, with "level": 50)


`lib.sh` sets `set -e`, and `wait` returned the probe's exit code, 50, which `set -e` treats as failure. I expected this in Kubernetes, where a passing probe makes the pod `Failed`; it hit my own script first. Fix: `rc=0; wait "$pid" || rc=$?`. Aborted output: `errors/instrument-1-set-e-abort.json`.

Re-run:

```text
trial 1 alloc     0 MiB: exit 50 level 50  pre 2278  during 2391  post 2278
trial 1 alloc  2048 MiB: exit 50 level 50  pre 2278  during 4441  post 2280
trial 2 alloc     0 MiB: exit 50 level 50  pre 2280  during 2394  post 2281
trial 2 alloc  2048 MiB: exit 50 level 50  pre 2280  during 4442  post 2281
trial 3 alloc     0 MiB: exit 50 level 50  pre 2281  during 2394  post 2281
trial 3 alloc  2048 MiB: exit 50 level 50  pre 2281  during 4442  post 2281
conditions after: 49.60,2527MHz,23.50W,0%,0x0000000000000400
```

| Trial | Context only (MiB) | Context + 2048 (MiB) | Allocation alone (MiB) |
| --- | --- | --- | --- |
| 1 | 113 | 2163 | 2050 |
| 2 | 114 | 2162 | 2048 |
| 3 | 113 | 2161 | 2048 |

**Finding.** A known 2048 MiB reads as 2048 to 2050 MiB, three times out of three, and `post` is back within 2 MiB of `pre` every time. Whole-card subtraction resolves to a few MiB on a quiet desktop, so the VRAM column stays.

**Finding.** A bare CUDA context costs **113 to 114 MiB**, not the roughly 300 MB I'd assumed from fixture data; that was a framework's context, which loads far more kernels. A cluster component holding a context would still add many times the floor, so VRAM stays a yes or no.

### 7. Docker Desktop restarted the stopped fleet

The slate check after the run:

In [ ]:
bash bench/cluster-matrix/tools/slate-check.sh > bench/cluster-matrix/slate/01-instrument-after.txt
diff bench/cluster-matrix/slate/baseline.txt bench/cluster-matrix/slate/01-instrument-after.txt

< <fleet>-dind-1-1 docker:28-dind exited          (and the other 10)
> <fleet>-dind-1-1 docker:28-dind running          (and the other 10)
> cluster-matrix-probe:01


In [ ]:
docker inspect --format '{{.Name}} started={{.State.StartedAt}} restartPolicy={{.HostConfig.RestartPolicy.Name}}' $(docker ps -q --filter label=com.docker.compose.project=<fleet>)

/<fleet>-light-1 started=2026-10-29T16:08:25.47Z restartPolicy=always   (all 11 between 16:08:24 and 16:08:26Z)


Four seconds after my first `docker build` started. Docker Desktop's log:

In [ ]:
grep -hE 'idle\] (busy -> idle|idle -> busy|VM (stopped|started))|engine (stop|start)' \
  "$LOCALAPPDATA"/Docker/log/host/com.docker.backend.exe.log*

15:38:38Z idle  busy -> idle
15:44:37Z       engine stop requested
15:44:48Z idle  VM stopped
16:08:20Z idle  idle -> busy (was idle for 29m42s; reason: moby-client ... /grpc)
16:08:20Z       engine start requested
16:08:26Z       engine start complete
16:08:26Z idle  VM started
16:08:26Z idle  idle -> busy (reason: [1 container(s)])


**Finding.** With the fleet stopped, nothing was running, so Resource Saver stopped Docker Desktop's VM six minutes later. My build woke the engine, and a fresh engine start applies `restart: always` to every container, including ones stopped by hand. So `docker compose stop` only lasts until something touches Docker, and every Docker-based candidate would.

No job ran while it was up:

In [ ]:
docker logs --since 2026-10-29T16:08:00Z <runner> | grep -E 'Running job|completed with result'

(0 lines on all 8 runners)


So the instrument numbers stand; the fleet doesn't use the GPU anyway. The pause that holds:

In [ ]:
ids=$(docker ps -aq --filter label=com.docker.compose.project=<fleet>)
docker inspect --format '{{.Name}} {{.HostConfig.RestartPolicy.Name}}' $ids > data/cluster-matrix/2026-10-29/01-instrument/raw/fleet-restart-policy-before.txt
docker update --restart=no $ids
docker compose --profile dind1 --profile dind2 --profile dind3 stop

11 exited, 11 with restart policy "no"


To resume: `docker update --restart=always $ids`, then `start`.

The floor in finding 2 was measured with Docker's VM stopped. During the instrument runs it was running, and idle sat 13 to 16 MiB higher. That can't separate the VM from drift, so the next floor records the VM state.

### Clean up

Nothing to uninstall. The probe image and `bench/cluster-matrix/probe/bin/gpu_probe` stay as the instrument for every candidate; from here on `cluster-matrix-probe:01` is part of the expected state.

In [ ]:
bash bench/cluster-matrix/tools/slate-check.sh > bench/cluster-matrix/slate/01-instrument-after.txt
diff bench/cluster-matrix/slate/baseline.txt bench/cluster-matrix/slate/01-instrument-after.txt

< disk used /: 8645M
> disk used /: 8660M
> cluster-matrix-probe:01
< 2366 MiB, 8151 MiB, 0 %
> 2270 MiB, 8151 MiB, 0 %


## 02. Plain Docker: the nesting ladder

**Question:** what does a container need to reach the GPU on WSL2, and does it survive another level of nesting? If the device and libraries are enough without the toolkit, every cluster whose nodes are containers can be fixed with mounts.

**Answer:** `--gpus` does not nest. Three mounts do, at every depth I tried.

| Setup | |
| --- | --- |
| Run | 2026-10-29 |
| Driver | 621.01, CUDA 13.5 |
| Docker Desktop | 4.88.1.237512, engine 29.7.2 |
| Images | `cluster-matrix-probe:01`; `docker:28-dind` for the nested rungs |
| Data | [`02-docker-ladder/`](../../data/cluster-matrix/2026-10-29/02-docker-ladder/) |

### 1. Slate and floor

In [ ]:
bash bench/cluster-matrix/tools/slate-check.sh > bench/cluster-matrix/slate/02-docker-ladder-before.txt
diff bench/cluster-matrix/slate/01-instrument-after.txt bench/cluster-matrix/slate/02-docker-ladder-before.txt

< disk used /: 8660M
> disk used /: 8665M
< 2270 MiB, 8151 MiB, 0 %
> 2282 MiB, 8151 MiB, 0 %


In [ ]:
grep -hE 'idle\] (VM (stopped|started))' "$LOCALAPPDATA"/Docker/log/host/com.docker.backend.exe.log | tail -1
wsl -d Ubuntu -- bash -c "cd $T && OUT_DIR=$T/data/cluster-matrix/2026-10-29/02-docker-ladder/raw SAMPLES=30 INTERVAL=2 bash bench/cluster-matrix/probe/floor.sh live"

(no VM event since the 16:08:26Z start, so Docker's VM was running)
median 2284 MiB  IQR 2 MiB  range 2282-2292 MiB  spread 10 MiB


With Docker's VM running the spread was 10 MiB, against 1 MiB with it stopped in step 01. One pair of samples, not proof the VM is the cause.

### 2. The driver changed since my notes

In [ ]:
nvidia-smi.exe --query-gpu=driver_version,name --format=csv,noheader
nvidia-smi.exe | sed -n 3p

621.01, NVIDIA GeForce RTX 5060
| NVIDIA-SMI 621.01                 KMD Version: 621.01        CUDA UMD Version: 13.5     |


**Finding.** dev-machine.md records 616.56 and CUDA 13.4. I'd copied that into step 01 without measuring and have corrected it there. `slate-check.sh` now prints the driver, so an update between candidates shows in the diff.

### 3. Docker Desktop has its own NVIDIA runtime

In [ ]:
docker version --format 'client {{.Client.Version}} / server {{.Server.Version}} / {{.Server.Os}}'
docker info --format '{{json .Runtimes}} default={{.DefaultRuntime}}'
docker run --rm --privileged --pid=host alpine:3.20 nsenter -t 1 -m -- sh -c 'ls -l /dev/dxg; ls /usr/lib/wsl/lib | head -20; ls /usr/lib/wsl/drivers | head -3'

client 29.7.2 / server 29.7.2 / linux
runtimes: io.containerd.runc.v2, nvidia (path: nvidia-container-runtime), runc   default=runc
crw-rw-rw- 1 root root 10, 125 Oct 29 16:26 /dev/dxg
libcuda.so  libcuda.so.1  libcuda.so.1.1  libcudadebugger.so.1  libd3d12.so  libd3d12core.so
libdxcore.so  libnvcuvid.so  ...  libnvidia-gpucomp.so.620.67.01  libnvidia-ml.so.1  ...
1394.inf_amd64_ff5c4e8141fc4520  3ware.inf_amd64_408ceed6ec8ab6cd  ...  (the Windows driver store)


Full output in `raw/versions.txt`.

**Finding.** On Docker Desktop, `--gpus all` goes through Docker Desktop's own `nvidia-container-runtime`, not the toolkit in Ubuntu. Its VM has `/dev/dxg`, `/usr/lib/wsl/lib` and `/usr/lib/wsl/drivers` like any WSL2 distro.

### 4. The ladder, stock

`bench/cluster-matrix/tools/ladder.sh` checks each container for `/dev/dxg`, `libcuda.so.1` in `/usr/lib/wsl/lib`, the loader cache and the GPU mounts, then runs the probe with 2048 MiB. Dind rungs start `docker:28-dind --privileged` and load the probe with `docker save | docker exec -i cm-dind docker load`.

In [ ]:
bash bench/cluster-matrix/tools/ladder.sh data/cluster-matrix/2026-10-29/02-docker-ladder/raw

L1none      level=none dxg=0 libcuda_file=0 libcuda_in_ldconfig=0 gpu_mounts=0
L1          level=50   dxg=1 libcuda_file=0 libcuda_in_ldconfig=1 gpu_mounts=9
L1m         level=10   dxg=1 libcuda_file=1 libcuda_in_ldconfig=0 gpu_mounts=1
L2none      level=none dxg=0 libcuda_file=0 libcuda_in_ldconfig=0 gpu_mounts=0
L2          level=none
L2m-gpus    level=none dxg=1 libcuda_file=0 libcuda_in_ldconfig=0 gpu_mounts=1
L2m-mounts  level=10   dxg=1 libcuda_file=1 libcuda_in_ldconfig=0 gpu_mounts=1


Each rung and its error, word for word (`raw/<rung>.txt`):

| Rung | Command (outer → inner) | Result |
| --- | --- | --- |
| L1none | `docker run --rm -e NVIDIA_VISIBLE_DEVICES=void IMG` | `gpu_probe: error while loading shared libraries: libnvidia-ml.so.1: cannot open shared object file: No such file or directory` (exit 127). The negative control. |
| L1 | `docker run --rm --gpus all IMG` | **level 50** |
| L1m | `docker run --rm --device /dev/dxg -v /usr/lib/wsl/lib:/usr/lib/wsl/lib:ro -e LD_LIBRARY_PATH=/usr/lib/wsl/lib -e NVIDIA_VISIBLE_DEVICES=void IMG` | level 10, `nvml_init`: `Unable to load the NVML library` |
| L2none | `--privileged --gpus all docker:28-dind` → `docker run --rm IMG` | exit 127, same as L1none |
| L2 | `--privileged --gpus all docker:28-dind` → `docker run --rm --gpus all IMG` | `docker: Error response from daemon: could not select device driver "" with capabilities: [[gpu]]` |
| L2m-gpus | `--privileged --gpus all docker:28-dind` → L1m's mounts | exit 127: `libnvidia-ml.so.1: cannot open shared object file` |
| L2m-mounts | `--privileged --device /dev/dxg -v /usr/lib/wsl/lib:...:ro docker:28-dind` → L1m's mounts | level 10, `Unable to load the NVML library` |

What `--gpus all` injects (L1's `/proc/self/mountinfo`):

```text
mount /dev/dxg
mount /usr/bin/nvidia-smi
mount /usr/lib/wsl/drivers/nv_dispi.inf_amd64_20f49c6eb5c9b39b/libnvdxgdmal.so.1
mount /usr/lib/wsl/drivers/nv_dispi.inf_amd64_20f49c6eb5c9b39b/libnvidia-ml.so.1
mount /usr/lib/wsl/drivers/nv_dispi.inf_amd64_20f49c6eb5c9b39b/libnvidia-ml_loader.so
mount /usr/lib/wsl/drivers/nv_dispi.inf_amd64_20f49c6eb5c9b39b/libnvidia-ptxjitcompiler.so.1
mount /usr/lib/wsl/drivers/nv_dispi.inf_amd64_20f49c6eb5c9b39b/nvidia-smi
mount /usr/lib/x86_64-linux-gnu/libnvidia-gpucomp.so
mount /usr/lib/x86_64-linux-gnu/libnvidia-ml.so.1
```

**Finding.** L1m got further than level 10 suggests: the loader found `libnvidia-ml.so.1` in `/usr/lib/wsl/lib`, but `nvmlInit` couldn't load the real library. Those files are thin loaders. The real NVML is in the Windows driver store under `/usr/lib/wsl/drivers/nv_dispi.inf_amd64_<hash>/`, which the NVIDIA hook mounts in L1.
#22 listed three preconditions per boundary: `/dev/dxg`, `/usr/lib/wsl/lib` on the loader path, and something doing the injection. There's a fourth: **the driver store, `/usr/lib/wsl/drivers`**.

**Finding.** A dind daemon has no NVIDIA runtime, so `--gpus` inside it fails (L2). And `--gpus` on the outer container leaves nothing to pass down: the hook mounts nine files, not the directories, so the inner container has no `/usr/lib/wsl/lib` (L2m-gpus).

### 5. Tuned: add the driver store

In [ ]:
bash bench/cluster-matrix/tools/ladder.sh data/cluster-matrix/2026-10-29/02-docker-ladder/raw L1md L2md-mounts L2md-gpus

| Rung | Command (outer → inner) | Result |
| --- | --- | --- |
| L1md | `docker run --rm --device /dev/dxg -v /usr/lib/wsl/lib:/usr/lib/wsl/lib:ro -v /usr/lib/wsl/drivers:/usr/lib/wsl/drivers:ro -e LD_LIBRARY_PATH=/usr/lib/wsl/lib -e NVIDIA_VISIBLE_DEVICES=void IMG` | **level 50** |
| L2md-mounts | `docker run -d --privileged --device /dev/dxg -v /usr/lib/wsl/lib:...:ro -v /usr/lib/wsl/drivers:...:ro docker:28-dind` → L1md's flags | **level 50** |
| L2md-gpus | `--privileged --gpus all docker:28-dind` → L1md's flags | exit 127, `libnvidia-ml.so.1: cannot open shared object file` |

**Finding.** This is the result #22 called decisive. **A device and two directories, with the loader path set, give a working GPU with no NVIDIA toolkit anywhere**, and passed down by hand they survive a second level of nesting. So "zero layers" was wrong. What fails is the toolkit route: the hook doesn't pass on what the next layer needs, and a nested daemon has no hook.

For k3d, kind and minikube this predicts the stock result (no GPU in the node) and the fix to try: pass `/dev/dxg`, `/usr/lib/wsl/lib` and `/usr/lib/wsl/drivers` into the node container, then into the pod.

### 6. Every rung gave the same answer three times

In [ ]:
for n in 2 3; do
  bash bench/cluster-matrix/tools/ladder.sh data/cluster-matrix/2026-10-29/02-docker-ladder/raw/repeat$n L1none L1 L1m L2none L2 L2m-gpus L2m-mounts L1md L2md-mounts L2md-gpus
done

Both repeats matched exactly (`raw/ladder-repeats.txt`). Nothing is flaky.

### Clean up

The slate check found 18 new anonymous volumes:

In [ ]:
bash bench/cluster-matrix/tools/slate-check.sh > bench/cluster-matrix/slate/02-docker-ladder-after.txt
diff bench/cluster-matrix/slate/01-instrument-after.txt bench/cluster-matrix/slate/02-docker-ladder-after.txt
for v in <new volumes>; do docker volume inspect --format '{{.Name}} {{.CreatedAt}} labels={{json .Labels}}' $v; done

0671ef33... 2026-10-29T16:28:18Z labels={"com.docker.volume.anonymous":""}
... 18 in all, created 16:28 to 16:32Z


**Finding.** `docker:dind` declares `/var/lib/docker` as a volume, and `docker rm -f` without `-v` leaves it behind: six dind rungs times three runs. I fixed `ladder.sh` to use `docker rm -f -v` and removed exactly those 18:

In [ ]:
vols=$(for v in $(docker volume ls -q --filter label=com.docker.volume.anonymous); do
  c=$(docker volume inspect --format '{{.CreatedAt}}' $v)
  [[ "$c" > "2026-10-29T16:28:00" && "$c" < "2026-10-29T16:33:00" ]] && echo $v
done)
docker volume rm $vols
bash bench/cluster-matrix/tools/slate-check.sh > bench/cluster-matrix/slate/02-docker-ladder-after.txt
diff bench/cluster-matrix/slate/01-instrument-after.txt bench/cluster-matrix/slate/02-docker-ladder-after.txt

18
< disk used /: 8660M
> disk used /: 8681M
> ## Driver
> 621.01
< 2270 MiB, 8151 MiB, 0 %
> 2316 MiB, 8151 MiB, 0 %


Back to the baseline. The driver lines are new because the check prints them now. The fleet stayed stopped.

## 03. k3s in WSL2

**Question:** k3s runs directly in the Ubuntu distro, with no container node. Does stock k3s give a working GPU, can the scheduler ration it, and what does it cost?

**Answer:** level 50 stock. Rationing takes the device plugin plus a one-line patch.

| Setup | |
| --- | --- |
| Run | 2026-10-29 |
| Driver | 621.01, CUDA 13.5 |
| k3s | v1.36.5+k3s1 (stable channel), containerd 2.3.4-k3s1.36 |
| Toolkit in the distro | nvidia-container-toolkit 1.20.1-1 |
| Device plugin (tuned) | v0.18.2 |
| Install | `curl -sfL https://get.k3s.io \| K3S_NODE_NAME=gpu-lab sh -` |
| Data | [`03-k3s/`](../../data/cluster-matrix/2026-10-29/03-k3s/) |

Everything ran as root inside Ubuntu through `k3s kubectl`, so no kubeconfig touched Windows. Scripts, all in `bench/cluster-matrix/tools/`: `k3s.sh` (the phases), `k3s-plugin.sh`, `k3s-cycle.sh` (timed cold cycles) and `inner.sh` (the check each pod runs).

### 1. Before install

In [ ]:
docker save -o bench/cluster-matrix/probe/probe-01.tar cluster-matrix-probe:01
bash bench/cluster-matrix/tools/slate-check.sh > bench/cluster-matrix/slate/03-k3s-before.txt
diff bench/cluster-matrix/slate/02-docker-ladder-after.txt bench/cluster-matrix/slate/03-k3s-before.txt
wsl -d Ubuntu -- bash -c "cd $T && OUT_DIR=$T/data/cluster-matrix/2026-10-29/03-k3s/raw SAMPLES=30 INTERVAL=2 bash bench/cluster-matrix/probe/floor.sh pre-install"
wsl -d Ubuntu -- bash -c 'systemctl is-system-running; free -m | sed -n 2p; df -BM --output=used / | tail -1; nproc; curl -sI https://get.k3s.io | head -1'

(slate: only disk and GPU memory moved)
median 2327 MiB  IQR 2 MiB  range 2324-2327 MiB  spread 3 MiB
running
Mem:           32097         988       29839          26        1609       31108
8690M
10
HTTP/2 200


The probe image goes into k3s's own containerd as a 98,863,104-byte tarball.

### 2. Stock k3s: Ready in 10 s, NVIDIA runtime set up on its own

In [ ]:
O=$T/data/cluster-matrix/2026-10-29/03-k3s/raw
wsl -d Ubuntu -u root -- bash $T/bench/cluster-matrix/tools/k3s.sh vram $O pre-install
wsl -d Ubuntu -u root -- bash $T/bench/cluster-matrix/tools/k3s.sh install $O
wsl -d Ubuntu -u root -- bash $T/bench/cluster-matrix/tools/k3s.sh state $O stock-ready

vram pre-install: 2291 MiB
[INFO]  Using v1.36.5+k3s1 as release
...
[INFO]  systemd: Starting k3s
## node
gpu-lab   Ready    control-plane   5s    v1.36.5+k3s1   ...   containerd://2.3.4-k3s1.36
## allocatable nvidia.com/gpu: ''
## runtimeclasses
crun  lunatic  nvidia  nvidia-experimental  slight  spin  wasmedge  wasmer  wasmtime  wws
## containerd runtimes
44:[plugins.'io.containerd.cri.v1.runtime'.containerd.runtimes.'nvidia']
48:  BinaryName = "/usr/bin/nvidia-container-runtime"


From `raw/timeline.csv`: install done at +9.1 s, node Ready at +10.1 s, probe image imported at +13.3 s.

**Finding.** The toolkit is already in the distro, so stock k3s finds `nvidia-container-runtime`, adds an `nvidia` runtime to its containerd and creates an `nvidia` RuntimeClass with no flags from me. There's no device plugin, so `nvidia.com/gpu` is not a resource.

### 3. Stock pods: a working GPU the scheduler can't see

In [ ]:
wsl -d Ubuntu -u root -- bash $T/bench/cluster-matrix/tools/k3s.sh probe $O stock-gpulimit nvidia 1
wsl -d Ubuntu -u root -- bash $T/bench/cluster-matrix/tools/k3s.sh probe $O stock-default
wsl -d Ubuntu -u root -- bash $T/bench/cluster-matrix/tools/k3s.sh probe $O stock-nvidia nvidia

| Pod | Result |
| --- | --- |
| `nvidia.com/gpu: 1` | stays `Pending`: `0/1 nodes are available: 1 Insufficient nvidia.com/gpu.` |
| no RuntimeClass | `dxg=0` and no libraries: `gpu_probe: error while loading shared libraries: libnvidia-ml.so.1: cannot open shared object file` (exit 127) |
| `runtimeClassName: nvidia` | **level 50**, `dxg=1 drivers_dir=1 libcuda_in_ldconfig=1 gpu_mounts=143` |

Full output: `raw/probe-stock-*.txt`.

Two harness mistakes. My first two calls failed with `k3s.sh: line 28: $2: unbound variable`, because `lib.sh` sets `set -u` and the optional arguments had no defaults. And `inner.sh` ends with an `echo`, so the container exits 0 whatever level the probe reached; the pod reads `Completed` and the level comes from the logs. That's also why the "exit 50 makes a passing pod Failed" problem never showed up here.

**Finding.** The `nvidia` runtime mounts the whole driver store file by file (143 mounts, DLLs included) and sets `NVIDIA_VISIBLE_DEVICES=void` even though the pod asked for `all`. The GPU works anyway: usable, but not rationed.

### 4. VRAM: no k3s component holds a CUDA context

| When | Median of 20 samples (MiB) |
| --- | --- |
| before install | 2291 |
| stock, idle for 60 s | 2292 |
| tuned, with the device plugin running | 2286 |

**Finding.** +1 and -5 MiB sit inside this session's 3 MiB floor plus drift. A CUDA context costs 113 MiB (step 01), so neither k3s nor the device plugin holds one; the plugin uses NVML, which takes no VRAM.

### 5. Tuned: the published device plugin runs and does nothing

In [ ]:
wsl -d Ubuntu -u root -- bash $T/bench/cluster-matrix/tools/k3s-plugin.sh $O

The script downloads `https://raw.githubusercontent.com/NVIDIA/k8s-device-plugin/v0.18.2/deployments/static/nvidia-device-plugin.yml` (sha256 `9e915ef0…`), applies it as is, then patches in the RuntimeClass.

As published:

```text
daemon set "nvidia-device-plugin-daemonset" successfully rolled out
allocatable='' after 60 s
E1029 17:07:12.683619  1 factory.go:113] Incompatible strategy detected auto
E1029 17:07:12.683644  1 factory.go:114] If this is a GPU node, did you configure the NVIDIA Container Toolkit?
E1029 17:07:12.684204  1 main.go:185] error starting plugins: error getting plugins: unable to create plugins: failed to construct resource managers: invalid device discovery strategy
```

With `runtimeClassName: nvidia` added to the DaemonSet:

In [ ]:
k3s kubectl -n kube-system patch ds nvidia-device-plugin-daemonset --type=json \
  -p '[{"op":"add","path":"/spec/template/spec/runtimeClassName","value":"nvidia"}]'

allocatable=1
I1029 17:07:48.449838  1 server.go:141] Starting to serve 'nvidia.com/gpu' on /var/lib/kubelet/device-plugins/nvidia-gpu.sock
I1029 17:07:48.452234  1 server.go:148] Registered device plugin for 'nvidia.com/gpu' with Kubelet


`nvidia.com/gpu` became allocatable 4.3 s after the patch.

**Finding.** The published manifest has no `runtimeClassName` and k3s defaults to runc, so the plugin can't see the GPU. The rollout still reports success and the pod shows Running while it backs off and restarts; the only signs are the log and an empty `allocatable`. One line fixes it.

### 6. Tuned pods: rationed, plus the missing-RuntimeClass trap

In [ ]:
wsl -d Ubuntu -u root -- bash $T/bench/cluster-matrix/tools/k3s.sh probe $O tuned-gpu-rc nvidia 1
wsl -d Ubuntu -u root -- bash $T/bench/cluster-matrix/tools/k3s.sh probe $O tuned-gpu-norc '' 1

| Pod | Result |
| --- | --- |
| `runtimeClassName: nvidia` + `nvidia.com/gpu: 1` | **level 50**, scheduled against the one allocatable GPU |
| `nvidia.com/gpu: 1`, no RuntimeClass | scheduled and holding the GPU, but `dxg=0` and exit 127: `libnvidia-ml.so.1: cannot open shared object file` |

**Finding.** The second row is the worst case from [gpu-debug.md](../gpu-debug.md), reproduced on purpose: by every Kubernetes measure the pod has the GPU, and it has no driver. On k3s every GPU pod needs both the resource request and the RuntimeClass.

### 7. WSL stops an idle distro, and k3s with it

While checking the cluster I noticed every system pod had restarted:

In [ ]:
wsl -d Ubuntu -u root -- bash $T/bench/cluster-matrix/tools/k3s-restarts.sh

## k3s service
NRestarts=0
ExecMainStartTimestamp=Thu 2026-10-29 13:08:39 EDT
## sandbox changes
17:02:35Z  helm-install-traefik   SandboxChanged  Pod sandbox changed, it will be killed and re-created.
17:06:32Z  traefik, svclb-traefik, local-path-provisioner, coredns, metrics-server  SandboxChanged
17:08:45Z  svclb-traefik, nvidia-device-plugin, metrics-server, coredns, traefik, local-path  SandboxChanged


The k3s service had never crashed, but its PID was 210 and it had started at 13:08:39 local time:

In [ ]:
wsl -d Ubuntu -u root -- bash $T/bench/cluster-matrix/tools/wsl-boots.sh

## pid 1 started
Thu Oct 29 13:08:34 2026
## wsl.conf
[boot]
command = mount --make-rshared /
systemd=true


`.wslconfig` sets `memory=32GB`, `processors=10`, `swap=4GB` and `networkingMode=nat`, with no idle setting. Every restart lines up with a gap in my own calls: a `sleep 60` from Git Bash, or minutes between commands.

Test A, nothing attached:

In [ ]:
pid1() { wsl -d Ubuntu -u root -- ps -o lstart= -p 1; }
pid1; sleep 90; pid1

pid1 before: Thu Oct 29 13:09:53 2026
pid1 after 90 s idle: Thu Oct 29 13:11:31 2026


Test B, a session held open in the background:

In [ ]:
wsl -d Ubuntu -- sleep 200 &
wsl -d Ubuntu -u root -- systemctl show -p ActiveEnterTimestampMonotonic --value init.scope   # at start
sleep 90
wsl -d Ubuntu -u root -- systemctl show -p ActiveEnterTimestampMonotonic --value init.scope   # at end

start: systemd 912743196
end:   systemd 912743196


The restart counters still went up by one during B: my own `sleep 200` started the stopped distro. systemd's start time shows no restart after that.

**Finding.** **On stock WSL2, k3s only lives while a `wsl.exe` session is open on the distro.** Close the last terminal and within about a minute the distro stops, the cluster with it, and every pod is recreated on the next start. None of the k3s-on-WSL guides I followed mention this. From here on, every timing run happens inside one `wsl.exe` call.

### 8. Cold install to a verified GEMM in about 18 s

In [ ]:
wsl -d Ubuntu -u root -- bash $T/bench/cluster-matrix/tools/k3s-cycle.sh $T/data/cluster-matrix/2026-10-29/03-k3s/raw 3

Each cycle uninstalls and clears leftovers, installs, waits for Ready, imports the probe image, runs a `runtimeClassName: nvidia` pod to level 50, records memory and disk, and uninstalls.

| Cycle | Install (s) | To Ready (s) | Import (s) | Pod (s) | **Install to level 50 (s)** | Uninstall (s) | WSL RAM at level 50 (MiB) | Disk at level 50 (MiB) | `/var/lib/rancher` (MiB) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | 8.8 | 0.6 | 3.4 | 231.8 | none (harness race) | 4.7 | 1781 | 10201 | 1448 |
| 2 | 8.8 | 0.6 | 3.4 | 6.4 | **19.2** | 3.4 | 1558 | 9481 | 729 |
| 3 | 8.3 | 0.6 | 3.5 | 6.1 | **18.5** | 3.0 | 1546 | 9488 | 739 |
| 4 | 7.3 | 0.6 | 3.5 | 5.2 | **16.7** | 4.4 | 1568 | 9430 | 695 |

Before each cycle the distro used about 1183 MiB of RAM and 8670 to 8698 MiB of disk.

Cycle 1 failed in my harness, not in k3s:

```text
Error from server (Forbidden): error when creating "STDIN": pods "probe" is forbidden: error looking up service account default/default: serviceaccount "default" not found
```

On a cluster a few seconds old the `default` service account doesn't exist yet, and I had retried the wait but not the apply. I fixed it (`until kubectl get serviceaccount default`) and ran cycle 4 in its place. Cycle 4's logs overwrote cycle 1's files, so the error above comes from `raw/cycles-run.txt`.

**Finding.** `curl` to a verified GEMM in a pod takes **16.7 to 19.2 s**, about 8 s of it downloading k3s. While it runs, k3s costs about **+370 to 390 MiB of RAM** and **+730 to 810 MiB of disk** in the distro, mostly `/var/lib/rancher`, and no VRAM. Uninstall takes 3 to 5 s.

### Clean up

`k3s-cycle.sh` ends each cycle with `k3s-uninstall.sh` and `rm -rf /etc/rancher /var/lib/rancher`.

In [ ]:
bash bench/cluster-matrix/tools/slate-check.sh > bench/cluster-matrix/slate/03-k3s-after.txt
diff bench/cluster-matrix/slate/02-docker-ladder-after.txt bench/cluster-matrix/slate/03-k3s-after.txt

< disk used /: 8681M
> disk used /: 8675M
< 2316 MiB, 8151 MiB, 0 %
> 2283 MiB, 8151 MiB, 0 %


Back to baseline. `bench/cluster-matrix/probe/probe-01.tar` stays; it's part of the instrument.

## 04. k3d

**Question:** k3d is k3s with each node a Docker container. Step 02 says a container node gets no GPU unless `/dev/dxg`, `/usr/lib/wsl/lib` and `/usr/lib/wsl/drivers` are passed down by hand. Does that hold, and can the scheduler ration the GPU with no NVIDIA toolkit?

**Answer:** nothing stock. Level 50 with two node mounts and privileged pods.

| Setup | |
| --- | --- |
| Run | 2026-10-29 |
| Driver | 621.01, CUDA 13.5 |
| k3d | v5.9.0 (`k3d-windows-amd64.exe`, sha256 `49d0b9c7…` matches the release's `checksums.txt`) |
| k3s inside | v1.35.5+k3s1 (`rancher/k3s:v1.35.5-k3s1`, k3d's default), containerd 2.2.3-k3s1 |
| Docker Desktop | 4.88.1 |
| Device plugin (tuned) | v0.18.2 |
| Data | [`04-k3d/`](../../data/cluster-matrix/2026-10-29/04-k3d/) |

### 1. Before install, and getting k3d

In [ ]:
bash bench/cluster-matrix/tools/slate-check.sh > bench/cluster-matrix/slate/04-k3d-before.txt
diff bench/cluster-matrix/slate/03-k3s-after.txt bench/cluster-matrix/slate/04-k3d-before.txt
wsl -d Ubuntu -- bash -c "cd $T && OUT_DIR=$T/data/cluster-matrix/2026-10-29/04-k3d/raw SAMPLES=30 INTERVAL=2 bash bench/cluster-matrix/probe/floor.sh pre-install"
V=$(curl -sfL https://api.github.com/repos/k3d-io/k3d/releases/latest | grep -o '"tag_name": *"[^"]*"' | cut -d'"' -f4)
curl -sfL -o bench/cluster-matrix/bin/k3d.exe "https://github.com/k3d-io/k3d/releases/download/$V/k3d-windows-amd64.exe"
curl -sfL "https://github.com/k3d-io/k3d/releases/download/$V/checksums.txt" | grep 'k3d-windows-amd64.exe$'
sha256sum bench/cluster-matrix/bin/k3d.exe
bench/cluster-matrix/bin/k3d.exe version

(slate: only disk and GPU memory moved)
median 2323 MiB  IQR 8 MiB  range 2308-2347 MiB  spread 39 MiB
49d0b9c796f5b8ba6f58a1dd5469e8b7b34bf9b6ce9078633eedb2789680a034  _dist/k3d-windows-amd64.exe
49d0b9c796f5b8ba6f58a1dd5469e8b7b34bf9b6ce9078633eedb2789680a034 *bench/cluster-matrix/bin/k3d.exe
k3d version v5.9.0
k3s version v1.35.5-k3s1 (default)


The floor spread 39 MiB, the widest so far, because I was using the desktop. k3d's k3s is one minor version behind step 03's.

### 2. Stock k3d: a privileged node with `/dev/dxg` and nothing else

In [ ]:
export KUBECONFIG=$PWD/data/cluster-matrix/2026-10-29/04-k3d/raw/kubeconfig-A
bench/cluster-matrix/bin/k3d.exe cluster create cm --kubeconfig-update-default=false --kubeconfig-switch-context=false --wait
bench/cluster-matrix/bin/k3d.exe kubeconfig write cm --output "$KUBECONFIG"
bench/cluster-matrix/bin/k3d.exe image import cluster-matrix-probe:01 -c cm
kubectl get runtimeclass --no-headers -o custom-columns=N:.metadata.name
docker exec k3d-cm-server-0 sh -c 'ls -l /dev/dxg; ls /usr/lib/wsl; grep -c nvidia /var/lib/rancher/k3s/agent/etc/containerd/config.toml'
docker inspect k3d-cm-server-0 --format 'privileged={{.HostConfig.Privileged}} image={{.Config.Image}}'

Cluster 'cm' created successfully!            (28.6 s including image pulls; Ready 0.7 s later; import 8.8 s)
crun lunatic nvidia nvidia-experimental slight spin wasmedge wasmer wasmtime wws
crw-rw-rw- 1 root root 10, 125 Oct 29 17:54 /dev/dxg
ls: cannot access '/usr/lib/wsl': No such file or directory
0
privileged=true image=docker.io/rancher/k3s:v1.35.5-k3s1


**Finding.** k3d nodes run privileged, so Docker hands them every device in its VM, `/dev/dxg` included, but none of the WSL libraries, so k3s finds no NVIDIA runtime. It still creates the `nvidia` RuntimeClass, pointing at a handler that doesn't exist.

### 3. Stock pods: nothing works

My first attempt failed in my script, not in k3d:

```text
error: error reading /c/PROJECTS/infinity-bench/bench/cluster-matrix/tools/inner.sh: The system cannot find the path specified.
The Pod "probe-A-default" is invalid: metadata.name: Invalid value: "probe-A-default": a lowercase RFC 1123 subdomain must consist of lower case alphanumeric characters ...
```

With `MSYS_NO_PATHCONV=1`, Windows `kubectl.exe` can't read `/c/...` paths, and pod names must be lowercase. I fixed both in `kpod.sh` with `cygpath -w`, the manifest on stdin, and `tr A-Z a-z`.

In [ ]:
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/04-k3d/raw a-default - - none 60
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/04-k3d/raw a-rc nvidia - none 60
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/04-k3d/raw a-gpu nvidia 1 none 20
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/04-k3d/raw a-hostpath - - hostpath 60

| Pod | Result |
| --- | --- |
| default | `dxg=0`, exit 127: `libnvidia-ml.so.1: cannot open shared object file` |
| `runtimeClassName: nvidia` | never starts: `Failed to create pod sandbox: ... unable to get OCI runtime for sandbox ...: no runtime for "nvidia" is configured` |
| `nvidia.com/gpu: 1` | `0/1 nodes are available: 1 Insufficient nvidia.com/gpu.` |
| hostPath mounts (`/dev/dxg`, both WSL folders, privileged) | never starts: `MountVolume.SetUp failed for volume "wsllib" : hostPath type check failed: /usr/lib/wsl/lib is not a directory` |

### 4. k3d's `--gpus all` doesn't help on WSL2

In [ ]:
bench/cluster-matrix/bin/k3d.exe cluster delete cm
bench/cluster-matrix/bin/k3d.exe cluster create cm --gpus all --kubeconfig-update-default=false --kubeconfig-switch-context=false --wait
docker exec k3d-cm-server-0 sh -c 'ls -l /dev/dxg; ls -d /usr/lib/wsl/lib /usr/lib/wsl/drivers/*; grep -c nvidia /var/lib/rancher/k3s/agent/etc/containerd/config.toml'
docker inspect k3d-cm-server-0 --format 'DeviceRequests={{json .HostConfig.DeviceRequests}}'
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/04-k3d/raw b-rc nvidia - none 30
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/04-k3d/raw b-hostpath - - hostpath 30

crw-rw-rw- 1 root root 10, 125 Oct 29 17:54 /dev/dxg
ls: cannot access '/usr/lib/wsl/lib': No such file or directory
/usr/lib/wsl/drivers/nv_dispi.inf_amd64_20f49c6eb5c9b39b
0
DeviceRequests=[{"Driver":"","Count":-1,"DeviceIDs":null,"Capabilities":[["gpu"]],"Options":{}}]
b-rc        no runtime for "nvidia" is configured
b-hostpath  hostPath type check failed: /usr/lib/wsl/lib is not a directory


**Finding.** `--gpus all` is Docker's `--gpus` on the node container, so it does what L1 did in step 02: a few driver store files, no `/usr/lib/wsl/lib`, no runtime. k3d's GPU guide pairs the flag with a custom k3s image that has the toolkit built in; the flag alone gives a pod nothing.

### 5. Tuned: WSL folders mounted into the node, level 50

First try, k3d's volume syntax with `:ro`:

In [ ]:
bench/cluster-matrix/bin/k3d.exe cluster create cm -v /usr/lib/wsl/lib:/usr/lib/wsl/lib:ro@all -v /usr/lib/wsl/drivers:/usr/lib/wsl/drivers:ro@all ...

WARN failed to stat file/directory '/usr/lib/wsl/drivers:/usr/lib/wsl/drivers' volume mount '/usr/lib/wsl/drivers:/usr/lib/wsl/drivers:ro': please make sure it exists
FATA Failed Cluster Configuration Validation:  failed to validate volume mount '/usr/lib/wsl/drivers:/usr/lib/wsl/drivers:ro': volume mount destination doesn't appear to be an absolute path: 'ro' in '/usr/lib/wsl/drivers:/usr/lib/wsl/drivers:ro'


The Windows build of k3d splits a three-part spec wrongly and checks the source path on the Windows filesystem, where it doesn't exist. My `until kubectl get nodes` loop then waited on a cluster that was never created; I stopped it after 400 s and gave every later wait a limit. Log: `errors/C1-create-ro-parse.log`.

Without `:ro`:

In [ ]:
bench/cluster-matrix/bin/k3d.exe cluster create cm -v /usr/lib/wsl/lib:/usr/lib/wsl/lib@all -v /usr/lib/wsl/drivers:/usr/lib/wsl/drivers@all \
  --kubeconfig-update-default=false --kubeconfig-switch-context=false --wait
docker exec k3d-cm-server-0 sh -c 'echo "lib files: $(ls /usr/lib/wsl/lib | wc -l)  driver dirs: $(ls /usr/lib/wsl/drivers | wc -l)"'
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/04-k3d/raw c-hostpath - - hostpath 60
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/04-k3d/raw c-default - - none 60

WARN failed to stat file/directory '/usr/lib/wsl/lib' ... please make sure it exists     (harmless: Docker resolves it in its VM)
lib files: 23  driver dirs: 909
c-hostpath   level=50   dxg=1 libcuda_file=1 drivers_dir=1 gpu_mounts=3
c-default    level=none dxg=0 libcuda_file=0 drivers_dir=0 gpu_mounts=0   libnvidia-ml.so.1: cannot open shared object file


**Finding.** The GPU works three container boundaries deep (Docker Desktop's VM, the k3d node, the pod) with no NVIDIA toolkit or runtime anywhere: two `-v` flags on the cluster, three hostPath mounts on the pod.

### 6. Tuned: the device plugin makes it schedulable, still no toolkit

Same v0.18.2 manifest as step 03, plus a strategic-merge patch (`raw/plugin-hostpath-patch.yaml`) that gives the plugin the three mounts, `LD_LIBRARY_PATH` and a privileged security context in place of a RuntimeClass:

In [ ]:
kubectl apply -f - < data/cluster-matrix/2026-10-29/03-k3s/raw/nvidia-device-plugin-v0.18.2.yml
kubectl -n kube-system patch ds nvidia-device-plugin-daemonset --type=strategic --patch-file "$(cygpath -w data/cluster-matrix/2026-10-29/04-k3d/raw/plugin-hostpath-patch.yaml)"
kubectl -n kube-system rollout status ds/nvidia-device-plugin-daemonset --timeout=180s
kubectl get node k3d-cm-server-0 -o jsonpath='{.status.allocatable.nvidia\.com/gpu}'

daemon set "nvidia-device-plugin-daemonset" successfully rolled out
allocatable='1' after 2s
I1029 18:13:14.658144  1 server.go:141] Starting to serve 'nvidia.com/gpu' on /var/lib/kubelet/device-plugins/nvidia-gpu.sock
I1029 18:13:14.660712  1 server.go:148] Registered device plugin for 'nvidia.com/gpu' with Kubelet


Then the pods, and a rationing check with one pod holding the GPU:

In [ ]:
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/04-k3d/raw d-gpu-hostpath - 1 hostpath 60
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/04-k3d/raw d-gpu-nomounts - 1 none 60
kubectl run holder --image=docker.io/library/cluster-matrix-probe:01 --image-pull-policy=Never --restart=Never \
  --overrides='{"spec":{"containers":[{"name":"holder","image":"docker.io/library/cluster-matrix-probe:01","imagePullPolicy":"Never","command":["sleep","600"],"resources":{"limits":{"nvidia.com/gpu":"1"}}}]}}'
kubectl wait --for=condition=Ready pod/holder --timeout=60s
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/04-k3d/raw d-second - 1 hostpath 20

| Pod | Result |
| --- | --- |
| `nvidia.com/gpu: 1` + hostPath mounts | **level 50** |
| `nvidia.com/gpu: 1`, no mounts | scheduled and holding the GPU, but exit 127: `libnvidia-ml.so.1: cannot open shared object file` |
| a second `nvidia.com/gpu: 1` while `holder` has it | `0/1 nodes are available: 1 Insufficient nvidia.com/gpu.` |

**Finding.** The device plugin only needs NVML to count the card, and NVML only needs the same three mounts, so k3d gets real rationing with no toolkit. The price is security: every GPU pod, and the plugin, runs privileged with hostPath mounts, where step 03's runtime route needed neither. The second row is the same trap as on k3s: a pod that asks for the GPU but forgets the mounts is scheduled with nothing.

### 7. Cold start to a rationed level 50 pod in about a minute

In [ ]:
bash bench/cluster-matrix/tools/k3d-cycle.sh data/cluster-matrix/2026-10-29/04-k3d/raw 3

Each cycle first removes k3d's three images, so it pays for the pulls the way k3s paid for its download. Then it creates the cluster with the two `-v` flags, imports the probe image, installs the patched plugin, waits for `nvidia.com/gpu`, runs a `nvidia.com/gpu: 1` hostPath pod to level 50, and deletes the cluster. A `wsl -d Ubuntu -- sleep 3600` session stayed open throughout so the WSL VM couldn't idle out.

| Cycle | Create, with pulls (s) | Ready (s) | Import (s) | Plugin to allocatable (s) | Pod (s) | **Create to level 50 (s)** | Delete (s) | VRAM before / up / after (MiB) | WSL RAM before / up / after (MiB) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | 28.4 | 0.2 | 7.7 | 13.3 | 4.9 | **54.5** | 2.2 | 2283 / 2280 / 2280 | 1027 / 1668 / 1035 |
| 2 | 28.4 | 0.3 | 8.8 | 12.3 | 4.6 | **54.3** | 2.1 | 2278 / 2278 / 2278 | 1035 / 1693 / 1052 |
| 3 | 28.6 | 0.3 | 8.8 | 22.3 | 3.7 | **63.6** | 2.1 | 2278 / 2279 / 2278 | 1056 / 1723 / 1059 |

Images pulled each cycle: `rancher/k3s:v1.35.5-k3s1` 337 MB, `k3d-proxy:5.9.0` 106 MB and `k3d-tools:5.9.0` 35 MB, plus the device plugin image, pulled inside the node.

Reading VRAM inline with `wsl -d Ubuntu -- bash -c "source ...; ...\$(...)"` gave `vram_used: command not found` twenty times, the Git Bash and `wsl.exe` quoting gotcha again, so it moved into `bench/cluster-matrix/tools/vram.sh`.

**Finding.** Nothing to a rationed, verified GEMM takes **54 to 64 s**, about three times k3s. Half of it is image pulls, which a second start would skip. No VRAM is held: the medians move at most 3 MiB against a 39 MiB floor. The cluster adds about **640 to 670 MiB** to the WSL VM, which Docker Desktop's VM shares with every distro. Delete takes about 2 s and leaves nothing behind.

### Clean up

In [ ]:
bench/cluster-matrix/bin/k3d.exe cluster list
docker rmi rancher/k3s:v1.35.5-k3s1 ghcr.io/k3d-io/k3d-proxy:5.9.0 ghcr.io/k3d-io/k3d-tools:5.9.0
rm -f bench/cluster-matrix/bin/k3d.exe; rmdir bench/cluster-matrix/bin
bash bench/cluster-matrix/tools/slate-check.sh > bench/cluster-matrix/slate/04-k3d-after.txt
diff bench/cluster-matrix/slate/03-k3s-after.txt bench/cluster-matrix/slate/04-k3d-after.txt
ls -A ~/.kube

NAME   SERVERS   AGENTS   LOADBALANCER
3
< disk used /: 8675M
> disk used /: 8679M
cache/
retired-2026-10-29/


`k3d cluster delete` removed its containers, network and volumes, but `~/.kube/cache` had appeared: Windows `kubectl` writes a discovery cache on first contact with a server, whatever `KUBECONFIG` says, and the slate check didn't look there. I removed it (it regenerates) and made `slate-check.sh` list `~/.kube` too:

In [ ]:
rm -rf ~/.kube/cache
bash bench/cluster-matrix/tools/slate-check.sh > bench/cluster-matrix/slate/04-k3d-after.txt
diff bench/cluster-matrix/slate/03-k3s-after.txt bench/cluster-matrix/slate/04-k3d-after.txt

< disk used /: 8675M
> disk used /: 8679M
> ## ~/.kube
< 2283 MiB, 8151 MiB, 0 %
> 2273 MiB, 8151 MiB, 0 %


Back to the baseline.

## 05. kind

**Question:** kind is the other container-node cluster, upstream Kubernetes rather than k3s. Does it fail like k3d, and does the same fix work?

**Answer:** yes to both. The mounts go in through `extraMounts`.

| Setup | |
| --- | --- |
| Run | 2026-10-29 |
| Driver | 621.01, CUDA 13.5 |
| kind | v0.33.0 (`kind-windows-amd64`, sha256 `4b22adaa…` matches the published `.sha256sum`) |
| Node image | `kindest/node:v1.37.0@sha256:a1ed56cf…`, 1.34 GB, Kubernetes v1.37.0, containerd 2.3.4, Debian 13 |
| Docker Desktop | 4.88.1 |
| Device plugin (tuned) | v0.18.2 |
| Data | [`05-kind/`](../../data/cluster-matrix/2026-10-29/05-kind/) |

### 1. Blank slate, noise floor, download

In [ ]:
bash bench/cluster-matrix/tools/slate-check.sh > bench/cluster-matrix/slate/05-kind-before.txt
diff bench/cluster-matrix/slate/04-k3d-after.txt bench/cluster-matrix/slate/05-kind-before.txt
wsl -d Ubuntu -- bash -c "cd $T && OUT_DIR=$T/data/cluster-matrix/2026-10-29/05-kind/raw SAMPLES=30 INTERVAL=2 bash bench/cluster-matrix/probe/floor.sh pre-install"
V=$(curl -sfL https://api.github.com/repos/kubernetes-sigs/kind/releases/latest | grep -o '"tag_name": *"[^"]*"' | cut -d'"' -f4)
curl -sfL -o bench/cluster-matrix/bin/kind.exe "https://github.com/kubernetes-sigs/kind/releases/download/$V/kind-windows-amd64"
curl -sfL "https://github.com/kubernetes-sigs/kind/releases/download/$V/kind-windows-amd64.sha256sum"
sha256sum bench/cluster-matrix/bin/kind.exe
bench/cluster-matrix/bin/kind.exe version

(slate: only disk and GPU memory moved)
median 2263 MiB  IQR 1 MiB  range 2262-2264 MiB  spread 2 MiB
4b22adaa135368c5a465d56bbd8e520cbea87272a06ca00b6078e7b81515c9fc  kind-windows-amd64
4b22adaa135368c5a465d56bbd8e520cbea87272a06ca00b6078e7b81515c9fc *bench/cluster-matrix/bin/kind.exe
kind v0.33.0 go1.26.7 windows/amd64


### 2. Stock: fails like k3d, no RuntimeClass

In [ ]:
export KUBECONFIG="$(cygpath -w $PWD/data/cluster-matrix/2026-10-29/05-kind/raw)\\kubeconfig-A"
bench/cluster-matrix/bin/kind.exe create cluster --name cm --kubeconfig "$KUBECONFIG" --wait 120s
bench/cluster-matrix/bin/kind.exe load docker-image cluster-matrix-probe:01 --name cm
kubectl get runtimeclass --no-headers
docker exec cm-control-plane sh -c 'ls -l /dev/dxg; ls /usr/lib/wsl; grep -c nvidia /etc/containerd/config.toml'
docker inspect cm-control-plane --format 'privileged={{.HostConfig.Privileged}} image={{.Config.Image}}'

(create 60.1 s including the 1.34 GB node image pull; Ready 0.2 s later; load 7.5 s)
cm-control-plane   Ready   control-plane   31s   v1.37.0   ...   Debian GNU/Linux 13 (trixie)   ...   containerd://2.3.4
No resources found
crw-rw-rw- 1 root root 10, 125 Oct 29 18:27 /dev/dxg
ls: cannot access '/usr/lib/wsl': No such file or directory
0
privileged=true image=kindest/node:v1.37.0@sha256:a1ed56cfb0e7b93589bdf97c8cd566405a265939e3620fc4f5de89adff580ae5


In [ ]:
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/05-kind/raw a-default - - none 60
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/05-kind/raw a-gpu - 1 none 20
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/05-kind/raw a-hostpath - - hostpath 60

| Pod | What happened |
| --- | --- |
| default | `dxg=0`, exit 127: `libnvidia-ml.so.1: cannot open shared object file` |
| `nvidia.com/gpu: 1` | `0/1 nodes are available: 1 Insufficient nvidia.com/gpu.` |
| hostpath | never starts: `MountVolume.SetUp failed for volume "wsldrivers" : hostPath type check failed` |

**Finding.** Privileged node with `/dev/dxg`, but no WSL libraries and no NVIDIA runtime. kind creates no RuntimeClasses, so there is no dangling `nvidia` handler as on k3s. In a stock pod the three-bit vector (`dxg`, libs, injection) is `0,0,0`, as on k3d.

### 3. Tuned: `extraMounts`, hostpath pods, patched plugin

kind's config file does what k3d's `-v` did, and on Windows it takes the Linux paths with no parse error or stat warning:

```yaml
# raw/kind-wsl-mounts.yaml
kind: Cluster
apiVersion: kind.x-k8s.io/v1alpha4
nodes:
- role: control-plane
  extraMounts:
  - {hostPath: /usr/lib/wsl/lib, containerPath: /usr/lib/wsl/lib, readOnly: true}
  - {hostPath: /usr/lib/wsl/drivers, containerPath: /usr/lib/wsl/drivers, readOnly: true}
```

In [ ]:
bench/cluster-matrix/bin/kind.exe delete cluster --name cm --kubeconfig "<kubeconfig-A>"
bench/cluster-matrix/bin/kind.exe create cluster --name cm --config "$(cygpath -w data/cluster-matrix/2026-10-29/05-kind/raw/kind-wsl-mounts.yaml)" --kubeconfig "$KUBECONFIG" --wait 120s
bench/cluster-matrix/bin/kind.exe load docker-image cluster-matrix-probe:01 --name cm
docker exec cm-control-plane sh -c 'echo "lib files: $(ls /usr/lib/wsl/lib | wc -l)  driver dirs: $(ls /usr/lib/wsl/drivers | wc -l)"'
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/05-kind/raw b-hostpath - - hostpath 60
kubectl apply -f - < data/cluster-matrix/2026-10-29/03-k3s/raw/nvidia-device-plugin-v0.18.2.yml
kubectl -n kube-system patch ds nvidia-device-plugin-daemonset --type=strategic --patch-file "$(cygpath -w data/cluster-matrix/2026-10-29/04-k3d/raw/plugin-hostpath-patch.yaml)"
kubectl get node cm-control-plane -o jsonpath='{.status.allocatable.nvidia\.com/gpu}'
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/05-kind/raw b-gpu-hostpath - 1 hostpath 60
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/05-kind/raw b-gpu-nomounts - 1 none 60
kubectl run holder ... (sleep 600, nvidia.com/gpu: 1); kubectl wait --for=condition=Ready pod/holder
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/05-kind/raw b-second - 1 hostpath 20

(create 30.8 s, image already pulled; load 7.5 s)
lib files: 23  driver dirs: 909
b-hostpath       level=50   dxg=1 libcuda_file=1 drivers_dir=1 gpu_mounts=3
daemon set "nvidia-device-plugin-daemonset" successfully rolled out
allocatable='1' after 3s
I1029 18:30:53.016557  1 server.go:148] Registered device plugin for 'nvidia.com/gpu' with Kubelet
b-gpu-hostpath   level=50   dxg=1 libcuda_file=1 drivers_dir=1 gpu_mounts=3
b-gpu-nomounts   level=none dxg=0 ...   libnvidia-ml.so.1: cannot open shared object file
b-second         0/1 nodes are available: 1 Insufficient nvidia.com/gpu.


**Finding.** The k3d fix works unchanged, first try: same node mounts, pod mounts and plugin patch. Pods reach level 50, the GPU is rationed, and a pod without the mounts is scheduled with nothing in it. This is how container nodes behave on WSL2, not something specific to k3s or k3d.

### 4. Nothing to a rationed level-50 pod: 71 to 80 s

`bench/cluster-matrix/tools/kind-cycle.sh` (from `k3d-cycle.sh`) removes `kindest/node` so it is pulled again, creates the cluster with the mounts config, loads the probe, installs the patched plugin, runs a rationed hostpath pod to level 50, and deletes everything, with a `wsl.exe` session open throughout.

I caught one bug before the run: `kind delete cluster` had no `--kubeconfig`, so it would have edited `~/.kube/config`.

In [ ]:
bash bench/cluster-matrix/tools/kind-cycle.sh data/cluster-matrix/2026-10-29/05-kind/raw 3

| Cycle | Create, with pull (s) | Ready (s) | Load (s) | Plugin to allocatable (s) | Pod (s) | **Create to level 50 (s)** | Delete (s) | VRAM before / up / after (MiB) | WSL RAM before / up / after (MiB) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | 45.5 | 0.3 | 7.6 | 14.2 | 3.6 | **71.2** | 1.3 | 2223 / 2221 / 2221 | 1059 / 1687 / 1066 |
| 2 | 49.1 | 0.3 | 7.6 | 13.2 | 3.6 | **73.8** | 1.2 | 2216 / 2224 / 2218 | 1074 / 1684 / 1061 |
| 3 | 55.6 | 0.3 | 7.5 | 13.1 | 3.6 | **80.1** | 1.3 | 2219 / 2219 / 2219 | 1064 / 1663 / 1063 |

**Finding.** **71 to 80 s**, the slowest so far, because the node image is 1.34 GB against k3d's 337 MB. No VRAM held: medians within 8 MiB, against a 2 MiB floor and a one-minute drift of the same size. About **600 to 630 MiB** of RAM while up, close to k3d. Delete is the fastest yet at 1.3 s.

### Clean up

In [ ]:
bench/cluster-matrix/bin/kind.exe get clusters
docker rmi $(docker images -q kindest/node)
rm -f bench/cluster-matrix/bin/kind.exe; rmdir bench/cluster-matrix/bin
rm -rf ~/.kube/cache
bash bench/cluster-matrix/tools/slate-check.sh > bench/cluster-matrix/slate/05-kind-after.txt
diff bench/cluster-matrix/slate/04-k3d-after.txt bench/cluster-matrix/slate/05-kind-after.txt

No kind clusters found.
1
> kind                        (under Docker networks)


**Finding.** `kind delete cluster` leaves the `kind` Docker network behind on purpose, since every kind cluster on the machine shares it.

In [ ]:
docker network inspect kind --format 'containers={{len .Containers}} created={{.Created}}'
docker network rm kind
bash bench/cluster-matrix/tools/slate-check.sh > bench/cluster-matrix/slate/05-kind-after.txt
diff bench/cluster-matrix/slate/04-k3d-after.txt bench/cluster-matrix/slate/05-kind-after.txt

containers=0 created=2026-10-29 18:27:05.784355804 +0000 UTC
kind
< disk used /: 8679M
> disk used /: 8675M
< 2273 MiB, 8151 MiB, 0 %
> 2187 MiB, 8151 MiB, 0 %


Back to baseline. Windows `kubectl` had recreated `~/.kube/cache`; I removed it above.

## 06. minikube

**Question:** an earlier minikube attempt ended at "NVML init failed inside the device plugin", on its documented GPU path, `--gpus=all`. What exactly fails, and can it be fixed?

**Answer:** level 10 stock, and the documented path fails the same way. One extra mount gets it to level 50.

| Setup | |
| --- | --- |
| Run | 2026-10-29 |
| Driver | 621.01, CUDA 13.5 |
| minikube | v1.39.0 (`minikube-windows-amd64.exe`, sha256 `77638646…` matches the published `.sha256`), Docker driver |
| Node image | `gcr.io/k8s-minikube/kicbase:v0.0.51`, 508 MiB, Debian 12, Kubernetes v1.37.0 |
| Device plugin | minikube's own, `nvcr.io/nvidia/k8s-device-plugin:v0.20.0` |
| Toolkit in the node | NVIDIA Container Toolkit CLI 1.20.0 |
| Docker Desktop | 4.88.1 |
| Data | [`06-minikube/`](../../data/cluster-matrix/2026-10-29/06-minikube/) |

minikube writes its kubeconfig to `KUBECONFIG`; I set `LC_ALL=C` so it answers in English.

### 1. Blank slate, noise floor, which minikube

In [ ]:
bash bench/cluster-matrix/tools/slate-check.sh > bench/cluster-matrix/slate/06-minikube-before.txt
diff bench/cluster-matrix/slate/05-kind-after.txt bench/cluster-matrix/slate/06-minikube-before.txt
wsl -d Ubuntu -- bash -c "cd $T && OUT_DIR=$T/data/cluster-matrix/2026-10-29/06-minikube/raw SAMPLES=30 INTERVAL=2 bash bench/cluster-matrix/probe/floor.sh pre-install"
ls -d ~/.minikube; LC_ALL=C minikube version; ls -d ~/.minikube
curl -sfL https://api.github.com/repos/kubernetes/minikube/releases/latest | grep -o '"tag_name": *"[^"]*"'

(slate: only disk and GPU memory moved)
median 2190 MiB  IQR 2 MiB  range 2190-2192 MiB  spread 2 MiB
ls: cannot access '/c/Users/<me>/.minikube': No such file or directory
minikube version: v1.38.1
/c/Users/<me>/.minikube/
"tag_name": "v1.39.0"


The installed minikube is v1.38.1 from February, and even `minikube version` recreates `~/.minikube` (see step 00). Every other candidate ran its latest release, so I downloaded v1.39.0 into `bench/cluster-matrix/bin/` and left the installed one alone.

In [ ]:
curl -sfL -o bench/cluster-matrix/bin/minikube.exe "https://github.com/kubernetes/minikube/releases/download/v1.39.0/minikube-windows-amd64.exe"
curl -sfL "https://github.com/kubernetes/minikube/releases/download/v1.39.0/minikube-windows-amd64.exe.sha256"
sha256sum bench/cluster-matrix/bin/minikube.exe

776386465ded2cf610ae397fe302de32c44d12134b5ce7ce98ab05bd7713b360
776386465ded2cf610ae397fe302de32c44d12134b5ce7ce98ab05bd7713b360 *bench/cluster-matrix/bin/minikube.exe


### 2. Stock: no GPU, but the toolkit is in the node

In [ ]:
export KUBECONFIG="$(cygpath -w $PWD/data/cluster-matrix/2026-10-29/06-minikube/raw)\\kubeconfig-A"
bench/cluster-matrix/bin/minikube.exe start -p cm --driver=docker
bench/cluster-matrix/bin/minikube.exe -p cm image load cluster-matrix-probe:01
docker exec cm sh -c 'ls -l /dev/dxg; ls /usr/lib/wsl; command -v nvidia-container-runtime nvidia-ctk'
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/06-minikube/raw a-default - - none 60
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/06-minikube/raw a-gpu - 1 none 20
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/06-minikube/raw a-hostpath - - hostpath 60

* Pulling base image v0.0.51 ...
* Downloading Kubernetes v1.37.0 preload ...
* Preparing Kubernetes v1.37.0 on containerd 2.3.4 ...     (start 80.6 s cold; load 9.6 s)
crw-rw-rw- 1 root root 10, 125 Oct 29 18:44 /dev/dxg
ls: cannot access '/usr/lib/wsl': No such file or directory
/usr/bin/nvidia-container-runtime
a-default   dxg=0, exit 127: libnvidia-ml.so.1: cannot open shared object file
a-gpu       0/1 nodes are available: 1 Insufficient nvidia.com/gpu.
a-hostpath  MountVolume.SetUp failed for volume "wsldrivers" : hostPath type check failed


**Finding.** Fails like k3d and kind: privileged node, `/dev/dxg` present, no WSL libraries. The default runtime is now containerd. Unlike the others, the node image ships `nvidia-container-runtime`.

### 3. The documented GPU path fails the same way

minikube's GPU handbook says `--container-runtime=docker --gpus=all`.

In [ ]:
bench/cluster-matrix/bin/minikube.exe delete -p cm
export KUBECONFIG="$(cygpath -w $PWD/data/cluster-matrix/2026-10-29/06-minikube/raw)\\kubeconfig-B"
bench/cluster-matrix/bin/minikube.exe start -p cm --driver=docker --container-runtime=docker --gpus=all
bench/cluster-matrix/bin/minikube.exe -p cm addons list | grep -i nvidia
kubectl get pods -A | grep nvidia
kubectl get node cm -o jsonpath='{.status.allocatable.nvidia\.com/gpu}'
kubectl -n kube-system logs -l name=nvidia-device-plugin-ds --tail=12

  - Using image nvcr.io/nvidia/k8s-device-plugin:v0.20.0
* Enabled addons: storage-provisioner, default-storageclass         (start 56.3 s, images cached)
│ nvidia-device-plugin        │ cm      │ disabled   │ 3rd party (NVIDIA)                     │
kube-system   nvidia-device-plugin-daemonset-j5dhk   1/1     Running   0          5s
allocatable: ''
I1029 18:47:50.372612  1 plugin-manager.go:101] Detected platform: nvml
E1029 18:47:50.373701  1 factory.go:99] Failed to initialize NVML: Not Supported.
E1029 18:47:50.373731  1 factory.go:100] If this is a GPU node, did you set the docker default runtime to `nvidia`?
W1029 18:47:50.373750  1 factory.go:107] nvml init failed: Not Supported
I1029 18:47:50.373760  1 main.go:416] No devices found. Waiting indefinitely.


In [ ]:
docker exec cm sh -c 'docker info --format "{{.DefaultRuntime}}"; cat /etc/docker/daemon.json; nvidia-ctk --version | head -1; ls -d /usr/lib/wsl/lib /usr/lib/wsl/drivers/*'
docker inspect cm --format 'DeviceRequests={{json .HostConfig.DeviceRequests}}'
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/06-minikube/raw b-default - - none 60

nvidia
{"exec-opts":["native.cgroupdriver=cgroupfs"],...,"default-runtime":"nvidia","runtimes":{"nvidia":{"path":"/usr/bin/nvidia-container-runtime","runtimeArgs":null}}}
NVIDIA Container Toolkit CLI version 1.20.0
ls: cannot access '/usr/lib/wsl/lib': No such file or directory
/usr/lib/wsl/drivers/nv_dispi.inf_amd64_20f49c6eb5c9b39b
DeviceRequests=[{"Driver":"","Count":-1,"DeviceIDs":null,"Capabilities":[["gpu"]],"Options":{}}]
b-default   level=10   dxg=1 libcuda_file=0 drivers_dir=1 gpu_mounts=8   "failed_stage": "nvml_init", "error": "Not Supported"
  mount /usr/lib/wsl/drivers/nv_dispi.inf_amd64_20f49c6eb5c9b39b/libcuda.so.1.1
  mount .../libnvidia-ml.so.1  .../libnvidia-ml_loader.so  .../nvidia-smi   (8 in all)


**Finding.** Now I know the cause. `--gpus=all` is Docker's `--gpus` on the node container, so the node gets `/dev/dxg` and a few driver-store files but no `/usr/lib/wsl/lib`. minikube makes `nvidia` the node's default Docker runtime and deploys the device plugin, but the toolkit can only pass on that partial set, so NVML answers `Not Supported`. The plugin pod still reads `Running`, and `addons list` says the plugin is `disabled` while its DaemonSet runs. Not a nesting limit: the same missing directory as everywhere else.

### 4. Tuned: one flag, the cleanest container-node result

On the Docker driver, `--mount-string` is a plain Docker bind, so all of `/usr/lib/wsl` can go into the node:

In [ ]:
bench/cluster-matrix/bin/minikube.exe delete -p cm
export KUBECONFIG="$(cygpath -w $PWD/data/cluster-matrix/2026-10-29/06-minikube/raw)\\kubeconfig-C"
bench/cluster-matrix/bin/minikube.exe start -p cm --driver=docker --container-runtime=docker --gpus=all --mount --mount-string=/usr/lib/wsl:/usr/lib/wsl
docker inspect cm --format '{{range .Mounts}}{{.Type}} {{.Source}} -> {{.Destination}}{{"\n"}}{{end}}' | grep -i wsl
docker exec cm sh -c 'echo "lib files: $(ls /usr/lib/wsl/lib | wc -l)  driver dirs: $(ls /usr/lib/wsl/drivers | wc -l)"'
kubectl get node cm -o jsonpath='{.status.allocatable.nvidia\.com/gpu}'
kubectl -n kube-system logs -l name=nvidia-device-plugin-ds --tail=5

bind /usr/lib/wsl -> /usr/lib/wsl
lib files: 23  driver dirs: 909
allocatable='1' after 8s
I1029 18:51:15.077116  1 plugin-manager.go:101] Detected platform: wsl
I1029 18:51:15.077156  1 plugin-manager.go:50] Using device discovery strategy: nvml
I1029 18:51:15.093630  1 server.go:149] Registered device plugin for 'nvidia.com/gpu' with Kubelet


In [ ]:
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/06-minikube/raw c-gpu - 1 none 60
grep -E 'privileged|hostPath' data/cluster-matrix/2026-10-29/06-minikube/raw/pod-c-gpu.yaml
kubectl run holder ... (sleep 600, nvidia.com/gpu: 1); kubectl wait --for=condition=Ready pod/holder
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/06-minikube/raw c-second - 1 none 20

c-gpu      level=50   dxg=1 libcuda_file=0 drivers_dir=1 gpu_mounts=143
(pod spec: no privileged, no hostPath)
c-second   0/1 nodes are available: 1 Insufficient nvidia.com/gpu.


**Finding.** With the WSL folder in the node, minikube's own toolkit injects everything (143 mounts, as on k3s), its plugin detects `platform: wsl`, and a pod that just asks for `nvidia.com/gpu: 1` reaches level 50. No privileged pods, no hostPath, no patched plugin. Of the three container-node clusters, only here does the toolkit route work instead of being worked around.

### 5. The catch: every CUDA image gets the GPU

A pod with no GPU request also reached level 50:

In [ ]:
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/06-minikube/raw c-default - - none 60

c-default  level=50   dxg=1 libcuda_file=0 drivers_dir=1 gpu_mounts=143


So I ran the probe with the variable forced off, the probe with its default, and a plain image:

In [ ]:
kubectl run c-env-void  --image=docker.io/library/cluster-matrix-probe:01 ... env NVIDIA_VISIBLE_DEVICES=void, args 2048 0
kubectl run c-env-unset --image=docker.io/library/cluster-matrix-probe:01 ... no env, args 2048 0
kubectl run c-ubuntu --image=ubuntu:24.04 --restart=Never --command -- sh -c 'ls /dev/dxg; ls /usr/lib/wsl; echo NVIDIA_VISIBLE_DEVICES=$NVIDIA_VISIBLE_DEVICES'

== c-env-void (NVIDIA_VISIBLE_DEVICES void, image ENV says all) exit=127
/usr/local/bin/gpu_probe: error while loading shared libraries: libnvidia-ml.so.1: cannot open shared object file
== c-env-unset (NVIDIA_VISIBLE_DEVICES unset, image ENV says all) exit=50
  "level": 50,
== c-ubuntu (plain ubuntu:24.04, no NVIDIA env)
ls: cannot access '/dev/dxg': No such file or directory
ls: cannot access '/usr/lib/wsl': No such file or directory
NVIDIA_VISIBLE_DEVICES=


**Finding.** With `nvidia` as the node's *default* runtime, `NVIDIA_VISIBLE_DEVICES=all` in the environment is enough to get the GPU, and every `nvidia/cuda` base image sets it. So a pod built on a CUDA image skips the scheduler: `nvidia.com/gpu` rations requests, not access. k3s had a milder form (any pod with the RuntimeClass got the GPU). k3d and kind do not, because nothing injects unless the pod mounts the paths itself. Closing it means making the toolkit ignore the variable from unprivileged pods and switching the plugin to another device-list strategy. I did not try that within this step's budget; it stays open for the post.

### 6. Nothing to an unprivileged, rationed level-50 pod: about two minutes

`bench/cluster-matrix/tools/minikube-cycle.sh` (from the k3d one) runs `minikube delete --all --purge` and removes kicbase so everything is pulled again, starts with the tuned flags, loads the probe, waits for minikube's plugin, runs a `nvidia.com/gpu: 1` pod with no mounts to level 50, and deletes, with a `wsl.exe` session open throughout.

In [ ]:
bash bench/cluster-matrix/tools/minikube-cycle.sh data/cluster-matrix/2026-10-29/06-minikube/raw 3

| Cycle | Start, with pulls (s) | Ready (s) | Load (s) | Plugin to allocatable (s) | Pod (s) | **Start to level 50 (s)** | Delete (s) | VRAM before / up / after (MiB) | WSL RAM before / up / after (MiB) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | 85.1 | 0.6 | 11.3 | 6.6 | 3.6 | **107.2** | 12.9 | 2232 / 2231 / 2231 | 1053 / 1670 / 1080 |
| 2 | 94.1 | 1.3 | 10.9 | 6.5 | 3.6 | **116.4** | 13.0 | 2231 / 2231 / 2232 | 1055 / 1688 / 1071 |
| 3 | 90.5 | 0.6 | 11.2 | 7.3 | 3.7 | **113.3** | 13.0 | 2232 / 2197 / 2197 | 1069 / 1681 / 1073 |

Every cycle's log shows `Pulling base image v0.0.51` and `Downloading Kubernetes v1.37.0 preload`.

**Finding.** **107 to 116 s**, the slowest so far. Start alone is 85 to 94 s, pulling the 508 MiB kicbase image and the Kubernetes preload. Delete is also the slowest, at 13 s. About **620 MiB** of RAM while up, like k3d and kind. No VRAM held: the 35 MiB drop in cycle 3 came while the cluster was up and stayed after the delete, so it was the desktop.

### Clean up

In [ ]:
bench/cluster-matrix/bin/minikube.exe delete --all --purge
docker rmi $(docker images -q gcr.io/k8s-minikube/kicbase)
rm -f bench/cluster-matrix/bin/minikube.exe; rmdir bench/cluster-matrix/bin
rm -rf ~/.kube/cache
ls -d ~/.minikube
bash bench/cluster-matrix/tools/slate-check.sh > bench/cluster-matrix/slate/06-minikube-after.txt
diff bench/cluster-matrix/slate/05-kind-after.txt bench/cluster-matrix/slate/06-minikube-after.txt

* Kicbase images have not been deleted. To delete images run:
  - docker rmi gcr.io/k8s-minikube/kicbase:<none> gcr.io/k8s-minikube/kicbase:v0.0.51
2
ls: cannot access '/c/Users/<me>/.minikube': No such file or directory
< disk used /: 8675M
> disk used /: 8687M
< 2187 MiB, 8151 MiB, 0 %
> 2200 MiB, 8151 MiB, 0 %


**Finding.** `minikube delete --all --purge` says itself that it keeps the kicbase images; they have to go by hand. Its network and container go with the delete. The installed v1.38.1 stays, as part of the baseline.

## 07. Rancher Desktop

**Question:** Rancher Desktop runs k3s in a WSL distro of its own, the same shape as step 03 packaged as an app. Does that give a working GPU, and does it ration it?

**Answer:** level 50 through hostPath mounts. Left alone, it starts Kubernetes 1.25.

| Setup | |
| --- | --- |
| Run | 2026-10-29 |
| Driver | 621.01, CUDA 13.5 |
| Rancher Desktop | v1.24.0 (`Rancher.Desktop.Setup.1.24.0.msi`, 670,932,992 bytes, sha512 `6428f330…` matches the published `.sha512sum`), per-user install, containerd engine |
| k3s inside | v1.25.16+k3s4 (its choice), then v1.36.5+k3s1 (mine) |
| Device plugin (tuned) | v0.18.2 |
| Data | [`07-rancher-desktop/`](../../data/cluster-matrix/2026-10-29/07-rancher-desktop/) |

I put this step off because the installer looked like it needed admin. The MSI supports a per-user install, so it needed no elevated prompt and no one at the keyboard. I picked the containerd engine to leave Docker Desktop's Docker context alone.

### 1. Blank slate, noise floor, installer

In [ ]:
bash bench/cluster-matrix/tools/slate-check.sh > bench/cluster-matrix/slate/07-rancher-desktop-before.txt
diff bench/cluster-matrix/slate/08b-dd-kubeadm-after.txt bench/cluster-matrix/slate/07-rancher-desktop-before.txt
wsl -d Ubuntu -- bash -c "cd $T && OUT_DIR=$T/data/cluster-matrix/2026-10-29/07-rancher-desktop/raw SAMPLES=30 INTERVAL=2 bash bench/cluster-matrix/probe/floor.sh pre-install"
curl -sfL https://api.github.com/repos/rancher-sandbox/rancher-desktop/releases/latest     # tag and asset URLs
curl -sfL -o bench/cluster-matrix/bin/rd.msi "$U/Rancher.Desktop.Setup.1.24.0.msi"
curl -sfL "$U/Rancher.Desktop.Setup.1.24.0.msi.sha512sum"
sha512sum bench/cluster-matrix/bin/rd.msi

(slate: only disk and GPU memory moved)
median 2336 MiB  IQR 1 MiB  range 2336-2338 MiB  spread 2 MiB
latest: v1.24.0
6428f330...ed77e *Rancher.Desktop.Setup.1.24.0.msi
6428f330...ed77e *bench/cluster-matrix/bin/rd.msi


### 2. Per-user install, no admin

```powershell
Start-Process msiexec.exe -ArgumentList @('/i', "`"$msi`"", 'MSIINSTALLPERUSER=1', '/qn', '/norestart', '/l*v', "`"$r\msi-install.log`"") -Wait -PassThru
Test-Path "$env:LOCALAPPDATA\Programs\Rancher Desktop\resources\resources\win32\bin\rdctl.exe"
```

```text
exit=0 seconds=12.2
True
Property(S): MSIINSTALLPERUSER = 1
MSI (s) ... Windows Installer installed the product. Product Name: Rancher Desktop. Product Version: 1.24.0.
```

It installed in 12.2 s into `%LOCALAPPDATA%\Programs\Rancher Desktop` and did not launch itself.

### 3. Headless first start: 47 s, on Kubernetes 1.25

In [ ]:
rdctl start --help      # container-engine.name, kubernetes.enabled, kubernetes.version, application.start-in-background, no-modal-dialogs
rdctl start --container-engine.name=containerd --kubernetes.enabled --application.start-in-background --no-modal-dialogs
kubectl --context rancher-desktop get nodes -o wide
kubectl --context rancher-desktop get node -o jsonpath='{.items[0].metadata.creationTimestamp} ready-at={...Ready lastTransitionTime}'

time="2026-10-29T16:42:40-04:00" level=info msg="About to launch ...\\Rancher Desktop.exe --application.startInBackground=true --containerEngine.name containerd --kubernetes.enabled=true --no-modal-dialogs ..."
<hostname>   Ready   control-plane,master   ...   v1.25.16+k3s4   192.168.127.2   ...   Rancher Desktop WSL Distribution   ...   containerd://2.3.2
2026-10-29T20:43:17Z ready-at=2026-10-29T20:43:27Z


`rdctl start` stays attached to the app, so my wait loop after it never ran and I stopped the command; the cluster was fine. **47 s** from launch to a Ready node, cold (it downloaded k3s). The node is named after the Windows machine.

It brought up **Kubernetes v1.25.16**, out of support since 2023, though nothing asked for it:

In [ ]:
cat "$LOCALAPPDATA/rancher-desktop/logs/deploymentProfile.log"            # empty
Get-Item 'HKCU:\Software\Policies\Rancher Desktop', 'HKLM:\...', 'HKCU:\Software\Rancher Desktop', 'HKLM:\...'   # all absent
grep -oE '"kubernetes":\{"version":"[^"]*"' "$LOCALAPPDATA/rancher-desktop/settings.json"
grep -oE 'v1\.3[0-9]\.[0-9]+\+k3s[0-9]' "$LOCALAPPDATA/rancher-desktop/cache/k3s-versions.json" | sort -V | tail -3

"kubernetes":{"version":"1.25.16"
v1.36.5+k3s1
v1.37.0+k3s1
v1.37.1+k3s1


No deployment profile, no registry policy, and its own version list goes up to 1.37.1. A fresh install started headless picks 1.25.16 on its own.

### 4. Stock: hostpath works, nothing else does

In [ ]:
rdctl shell sh -c 'nerdctl --namespace k8s.io load -i /mnt/c/.../bench/cluster-matrix/probe/probe-01.tar; ls -l /dev/dxg; ls /usr/lib/wsl; echo "lib files: $(ls /usr/lib/wsl/lib | wc -l)"; command -v nvidia-container-runtime || echo "no nvidia-container-runtime"; head -2 /etc/os-release'
kubectl get runtimeclass
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/07-rancher-desktop/raw a-default - - none 60
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/07-rancher-desktop/raw a-rc nvidia - none 30
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/07-rancher-desktop/raw a-gpu - 1 none 20
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/07-rancher-desktop/raw a-hostpath - - hostpath 60

Loaded image: cluster-matrix-probe:01
crw-rw-rw-    1 root     0          10, 125 Oct 29 16:42 /dev/dxg
drivers
lib
lib files: 21
no nvidia-container-runtime
NAME="Rancher Desktop WSL Distro"
(no runtimeclasses)


| Pod | What happened |
| --- | --- |
| default | `dxg=0`, exit 127: `libnvidia-ml.so.1: cannot open shared object file` |
| `runtimeClassName: nvidia` | `pod rejected: RuntimeClass "nvidia" not found` |
| `nvidia.com/gpu: 1` | `Insufficient nvidia.com/gpu` |
| hostpath | **level 50** on the stock cluster |

**Finding.** The node is a WSL distro, so WSL gives it `/dev/dxg` and `/usr/lib/wsl` as it does Ubuntu: 21 library files against 23 in Ubuntu and Docker Desktop's VM, enough for the probe. The distro has no NVIDIA toolkit and so no RuntimeClass (k3s in step 03 got the toolkit from Ubuntu). That puts it where Docker Desktop's kubeadm mode is.

### 5. `rdctl set` moves it to 1.36.5 in 26 s

In [ ]:
rdctl set --kubernetes.version=1.36.5
kubectl --context rancher-desktop get nodes -o wide
kubectl get runtimeclass
ls "$LOCALAPPDATA/rancher-desktop/cache/k3s"

Status: reconfiguring Rancher Desktop to apply changes (this may take a while).
v1.36 ready after 26s: v1.36.5+k3s1
<hostname>   Ready   control-plane,master   12m   v1.36.5+k3s1   ...   containerd://2.3.2
(no runtimeclasses)
v1.25.16+k3s4/  v1.36.5+k3s1/


It upgraded in place: same node, same loaded image. Still no RuntimeClasses on 1.36.5; Rancher Desktop does not ship the ones plain k3s creates.

### 6. Tuned: the hostpath plugin rations the GPU

In [ ]:
kubectl apply -f - < data/cluster-matrix/2026-10-29/03-k3s/raw/nvidia-device-plugin-v0.18.2.yml
kubectl -n kube-system patch ds nvidia-device-plugin-daemonset --type=strategic --patch-file "$(cygpath -w data/cluster-matrix/2026-10-29/04-k3d/raw/plugin-hostpath-patch.yaml)"
kubectl get node <hostname> -o jsonpath='{.status.allocatable.nvidia\.com/gpu}'
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/07-rancher-desktop/raw b-gpu-hostpath - 1 hostpath 60
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/07-rancher-desktop/raw b-gpu-nomounts - 1 none 30
kubectl run holder ... (sleep 600, nvidia.com/gpu: 1); kubectl wait --for=condition=Ready pod/holder
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/07-rancher-desktop/raw b-second - 1 hostpath 20

allocatable='1' after 10s
I1029 20:56:04.381179  1 server.go:148] Registered device plugin for 'nvidia.com/gpu' with Kubelet
b-gpu-hostpath   level=50   ...   Warning FailedScheduling 0/1 nodes are available: 1 Insufficient nvidia.com/gpu.
b-gpu-nomounts   level=none dxg=0 ...   libnvidia-ml.so.1: cannot open shared object file
b-second         0/1 nodes are available: 1 Insufficient nvidia.com/gpu.


**Finding.** Rationed and working with privileged hostpath pods, as on k3d, kind and Docker Desktop kubeadm. The `FailedScheduling` warning on `b-gpu-hostpath` is the scheduler working: `a-gpu` from the stock run was still `Pending` with its request, got the GPU when the plugin registered, and `b-gpu-hostpath` waited until it finished.

In [ ]:
wsl -d Ubuntu -- bash $T/bench/cluster-matrix/tools/vram.sh 20
du -sh "$LOCALAPPDATA/rancher-desktop" "$LOCALAPPDATA/Programs/Rancher Desktop"

rd up (k8s 1.36.5 + plugin): vram_mib=2222 wsl_mem_mib=1637
4.7G	.../AppData/Local/rancher-desktop
2.2G	.../AppData/Local/Programs/Rancher Desktop
distro-data vhdx MiB: 3404


No VRAM held. About 590 MiB of WSL RAM above this machine's ~1050 MiB idle. Disk is **6.9 GB** (2.2 GB app, 4.7 GB data), the most of any candidate.

### 7. Cold cycle: 66 s to a rationed level-50 pod, version flag ignored

`bench/cluster-matrix/tools/rd-cycle.sh` factory-resets with the k3s cache removed, starts with `--kubernetes.version=1.36.5`, loads the probe, installs the hostpath plugin and runs a rationed hostpath pod.

My first attempt was invalid twice over, both my mistakes (files in `errors/attempt1-*`):

```text
Error: unknown flag: --remove-kubernetes-cache            (rdctl reset --factory does not take it; the deprecated factory-reset does)
The DaemonSet "nvidia-device-plugin-daemonset" is invalid: ... cannot set `allowPrivilegeEscalation` to false and `privileged` to true
```

The reset never happened, so it measured the cluster already running (Ready in 0.2 s). And re-applying the published manifest over the existing DaemonSet restored its security context before my patch. I fixed both with `factory-reset --remove-kubernetes-cache` and by deleting the DaemonSet first.

In [ ]:
bash bench/cluster-matrix/tools/rd-cycle.sh data/cluster-matrix/2026-10-29/07-rancher-desktop/raw 1

| Start to Ready (s) | Import (s) | Plugin to allocatable (s) | Pod (s) | **Start to level 50 (s)** | k8s | VRAM before / up (MiB) | WSL RAM before / up (MiB) |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 42.8 | 3.4 | 16.8 | 2.9 | **65.9** | **v1.25.16+k3s4** | 2159 / 2183 | 717 / 1446 |

```text
time="2026-10-29T17:04:01-04:00" level=info msg="successfully cleared data."
"kubernetes":{"version":"1.25.16"            (settings.json after the run)
... --kubernetes.version 1.36.5 --kubernetes.enabled=true --no-modal-dialogs ...   (the launch line in cycle1-start.log)
```

**Finding.** **66 s** cold, k3s download included, from start to a rationed, verified GEMM. But on 1.25.16 again: on a fresh start Rancher Desktop takes `--kubernetes.version` on its command line and still writes 1.25.16 to its settings. Only `rdctl set` on a running instance changed it. One cycle only.

### Clean up

In [ ]:
rdctl factory-reset --remove-kubernetes-cache
tasklist | grep -ic "rancher desktop"
wsl -l -v

```powershell
Start-Process msiexec.exe -ArgumentList @('/x', "`"$msi`"", 'MSIINSTALLPERUSER=1', '/qn', '/norestart', '/l*v', "`"$r\msi-uninstall.log`"") -Wait -PassThru
# then Test-Path on every location step 00 found
```

In [ ]:
mv ~/.kube/config data/cluster-matrix/2026-10-29/07-rancher-desktop/raw/kubeconfig-rancher-desktop-written.yaml; rm -rf ~/.kube/cache
rm -f bench/cluster-matrix/bin/rd.msi
bash bench/cluster-matrix/tools/slate-check.sh > bench/cluster-matrix/slate/07-rancher-desktop-after.txt
diff bench/cluster-matrix/slate/08b-dd-kubeadm-after.txt bench/cluster-matrix/slate/07-rancher-desktop-after.txt

time="2026-10-29T17:04:01-04:00" level=info msg="successfully cleared data."
0
* Ubuntu  Stopped  2
  docker-desktop  Stopped  2
exit=0 seconds=3.1
gone: ...\AppData\Local\Programs\Rancher Desktop
gone: ...\AppData\Local\rancher-desktop
gone: ...\AppData\Roaming\rancher-desktop
gone: ...\.rd
left: C:\Program Files\Rancher Desktop  0 MiB        (the empty folder from before; part of the baseline)
< disk used /: 8731M
> disk used /: 8775M
< 2344 MiB, 8151 MiB, 0 %
> 2160 MiB, 8151 MiB, 0 %


**Finding.** A factory reset, then the uninstall, leaves nothing: both distros, the data and the app are gone. In step 00 an uninstall on its own left 1.2 GB and two distros behind on this machine. The order makes the difference.

## 08. Docker Desktop, kind mode

**Question:** Docker Desktop's built-in Kubernetes is the zero-install option. Does it give a GPU, and does it fail the same way kind does? If it does, that also checks the method.

**Answer:** only by copying the driver into every pod.

| Setup | |
| --- | --- |
| Run | 2026-10-29 |
| Driver | 621.01, CUDA 13.5 |
| Docker Desktop | 4.88.1.237512, CLI plugin v0.4.3 |
| Kubernetes | v1.35.5, kind mode, one node (`desktop-control-plane`, `kindest/node:v1.35.5`, containerd 2.3.1, Debian 13) |
| Device plugin (tuned) | v0.18.2 |
| Data | [`08-docker-desktop/`](../../data/cluster-matrix/2026-10-29/08-docker-desktop/) |

Docker Desktop was in kind mode (`docker desktop kubernetes status` says `Mode: kind`). kubeadm mode is step 08b.

### 1. Slate, noise floor, and the switch

In [ ]:
bash bench/cluster-matrix/tools/slate-check.sh > bench/cluster-matrix/slate/08-docker-desktop-before.txt
diff bench/cluster-matrix/slate/06-minikube-after.txt bench/cluster-matrix/slate/08-docker-desktop-before.txt
wsl -d Ubuntu -- bash -c "cd $T && OUT_DIR=$T/data/cluster-matrix/2026-10-29/08-docker-desktop/raw SAMPLES=30 INTERVAL=2 bash bench/cluster-matrix/probe/floor.sh pre-enable"
grep -oE '"Kubernetes[A-Za-z]*" *: *("[^"]*"|[a-z0-9]+)' "$APPDATA/Docker/settings-store.json"
docker desktop --help; docker desktop kubernetes --help
docker desktop kubernetes status

(slate: only disk and GPU memory moved)
median 2237 MiB  IQR 3 MiB  range 2237-2240 MiB  spread 3 MiB
"KubernetesEnabled": false
"KubernetesNodesCount": 1
"KubernetesNodesVersion": "1.35.5"
Commands (kubernetes): images, reset-cluster, status
State:              disabled
Mode:               kind
Node Count:         1
Version:            1.35.5


The `docker desktop` CLI can show and reset Kubernetes but not turn it on. The switch is `KubernetesEnabled` in `%APPDATA%\Docker\settings-store.json`, then a restart. I backed the file up to `raw/settings-store.before.json` and changed only that key.

### 2. Cold enable: 82 s to Ready

In [ ]:
cp "$APPDATA/Docker/settings-store.json" data/cluster-matrix/2026-10-29/08-docker-desktop/raw/settings-store.before.json
sed -i 's/"KubernetesEnabled": *false/"KubernetesEnabled": true/' "$APPDATA/Docker/settings-store.json"
docker desktop restart
docker desktop kubernetes status --format json
kubectl get node -o jsonpath='{.items[0].metadata.creationTimestamp} ready-at={.items[0].status.conditions[?(@.type=="Ready")].lastTransitionTime}'

restart issued 2026-10-29T19:37:55Z
{"content": {"mode": "kind", "nodeCount": 1, "progressMessage": "Kubernetes is up and running", ...}, "status": "running", ...}
2026-10-29T19:38:55Z ready-at=2026-10-29T19:39:17Z


**82 s** from restart to Ready, cold: no kind images were cached. It pulled about 2.2 GB: `kindest/node:v1.35.5` (1.28 GB), `docker/desktop-cloud-provider-kind:v0.6.0` (613 MB), `envoyproxy/envoy:v1.36.7` (253 MB) and `docker/desktop-containerd-registry-mirror:v0.0.4` (53 MB). It also wrote a `docker-desktop` context into `~/.kube/config` and made it current without asking, with no setting for where it goes.

My status loop grepped for `State` and spun until I stopped it. In the JSON the field is `status`.

### 3. Stock: same as kind

In [ ]:
export KUBECONFIG="$(cygpath -w ~/.kube/config)"
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/08-docker-desktop/raw a-default - - none 60

a-default  ErrImageNeverPull: Container image "docker.io/library/cluster-matrix-probe:01" is not present with pull policy of Never


The node is a kind container hidden from `docker ps`, so `kind load` is out. Images arrive through Docker Desktop's registry mirror, so I added `PULL=` to `kpod.sh` to set the pull policy.

In [ ]:
export PULL=IfNotPresent
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/08-docker-desktop/raw a-default - - none 90
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/08-docker-desktop/raw a-gpu - 1 none 20
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/08-docker-desktop/raw a-hostpath - - hostpath 60

| Pod | Result |
| --- | --- |
| default | `dxg=0 libcuda_file=0 drivers_dir=0`, exit 127: `libnvidia-ml.so.1: cannot open shared object file` |
| `nvidia.com/gpu: 1` | `0/1 nodes are available: 1 Insufficient nvidia.com/gpu.` |
| hostpath | `/dev/dxg` passes its CharDevice check; `hostPath type check failed: /usr/lib/wsl/lib is not a directory` (same for `/usr/lib/wsl/drivers`) |

No RuntimeClasses.

**Finding.** Exactly kind in step 05: a privileged node with `/dev/dxg`, no WSL libraries, no NVIDIA runtime, nothing for a stock pod. **kind and Docker Desktop's kind mode give the same vector**, the method check #22 asked for.

### 4. Tuned: the pod brings the driver

Docker Desktop has no setting for extra node mounts, so the k3d and kind fix is out and the files have to ship with the pod. Their size:

In [ ]:
docker run --rm -v /usr/lib/wsl:/w:ro alpine:3.20 sh -c 'du -sh /w/lib; ls -d /w/drivers/nv_*; du -sh /w/drivers/nv_dispi.inf_amd64_*; du -sh /w/drivers'

384.1M	/w/lib
2.9G	/w/drivers/nv_dispi.inf_amd64_20f49c6eb5c9b39b
78.0M	/w/drivers/nv_dispi.inf_amd64_adc55ecfca814224
4.0G	/w/drivers


Too much to carry whole, so I bundled only what the NVIDIA hook mounted in steps 02 and 06. From the driver directory, under its exact name: `libcuda.so.1.1 libcuda_loader.so libnvdxgdmal.so.1 libnvidia-ml.so.1 libnvidia-ml_loader.so libnvidia-ptxjitcompiler.so.1 libnvidia-gpucomp.so nvcubins.bin nvidia-smi`. Plus `/usr/lib/wsl/lib` minus its video, DirectX and OptiX libraries.

In [ ]:
docker run --rm -v /usr/lib/wsl:/w:ro -v "<bundle dir>":/out alpine:3.20 sh -c '<copy the files above>'
printf 'FROM alpine:3.20\nCOPY wsl /bundle/wsl\n' > Dockerfile
docker build -q -t cm-wsl-bundle:621.01 .

261.4M	/out/wsl/lib
277.8M	/out/wsl/drivers
size=170588332


The pod (`raw/pod-a-bundle.yaml`) has an init container that copies the bundle into an `emptyDir` at `/usr/lib/wsl`. The probe also gets `/dev/dxg` from the node, `LD_LIBRARY_PATH`, and privileged mode.

In [ ]:
kubectl apply -f - < data/cluster-matrix/2026-10-29/08-docker-desktop/raw/pod-a-bundle.yaml

dxg=1 libcuda_file=1 drivers_dir=1 gpu_mounts=1
  mount /dev/dxg
"level": 50


Then the device plugin, with the same bundle via an init container (`raw/plugin-bundle-patch.yaml`):

In [ ]:
kubectl apply -f - < data/cluster-matrix/2026-10-29/03-k3s/raw/nvidia-device-plugin-v0.18.2.yml
kubectl -n kube-system patch ds nvidia-device-plugin-daemonset --type=strategic --patch-file "$(cygpath -w data/cluster-matrix/2026-10-29/08-docker-desktop/raw/plugin-bundle-patch.yaml)"
kubectl get node desktop-control-plane -o jsonpath='{.status.allocatable.nvidia\.com/gpu}'

allocatable='1' after 4s
I1029 19:54:24.728639  1 server.go:148] Registered device plugin for 'nvidia.com/gpu' with Kubelet


**Finding.** Even the sealed kind node reaches a verified GEMM and a schedulable GPU, if every GPU pod carries 539 MB of driver files. I wouldn't run it: the bundle is a copy of driver 621.01, so it breaks on the next Windows driver update, and every GPU pod is privileged.

### 5. Warm start to a rationed level-50 pod: 81 s

`bench/cluster-matrix/tools/dd-cycle.sh` resets and disables Kubernetes, removes the four images, re-enables it, deploys the bundled plugin and a bundled `nvidia.com/gpu: 1` pod, and waits for level 50.

In [ ]:
bash bench/cluster-matrix/tools/dd-cycle.sh data/cluster-matrix/2026-10-29/08-docker-desktop/raw 1

| Restart to running (s) | To Ready (s) | Plugin to allocatable (s) | Pod (s) | **Restart to level 50 (s)** | VRAM before / up (MiB) | WSL RAM before / up (MiB) |
| --- | --- | --- | --- | --- | --- | --- |
| 46.4 | 0.2 | 28.1 | 6.1 | **80.9** | 2314.5 / 2316 | 1546 / 1776 |

The image removal failed, so this run was **warm**:

```text
Untagged: envoyproxy/envoy:v1.36.7
Error response from daemon: conflict: unable to delete kindest/node:v1.35.5 (must be forced) - container 4037e60884bb is using its referenced image ce977ae6d659
Error response from daemon: conflict: unable to delete docker/desktop-cloud-provider-kind:v0.6.0 (must be forced) - container 8fbb859fba64 is using its referenced image 46d51b990e97
Error response from daemon: conflict: unable to delete docker/desktop-containerd-registry-mirror:v0.0.4 (must be forced) - container a45a20107bbc is using its referenced image 1de142695bb0
```

Only envoy was pulled again; the containers holding the rest don't show in `docker ps -a` (finding 6). I kept it as a warm number. For cold, take the 82 s to Ready from finding 2 plus about 35 s for plugin and pod.

No VRAM held. The RAM figures aren't clean: restarts change Docker Desktop's own footprint, and the cluster was still running when I thought it was off.

### 6. "Disabled" isn't off

**The cluster survives being switched off.** Setting `KubernetesEnabled` to false and restarting reported `State: stopped`. Turning it back on brought back the same node (created 19:38:55), Ready in 8 s.

**"Stopped" still runs the containers.** The image removal above failed on containers I couldn't list. After disabling again:

In [ ]:
docker rmi kindest/node:v1.35.5     # to get the container IDs from the conflict errors
docker inspect --format '{{.ID}} {{.Name}} {{.State.Status}} {{.Config.Image}}' 7e063ae49e37 a1cce1bb1da6 ce43b719c26d

7e063ae4... /kind-cloud-provider    running docker.io/docker/desktop-cloud-provider-kind:v...
a1cce1bb... /kind-registry-mirror   running docker.io/docker/desktop-containerd-registry-...
ce43b719... /desktop-control-plane  running kindest/node:v1.35.5


The control-plane node was **running** while Docker Desktop said stopped, and none of the three show in `docker ps -a`. `docker desktop kubernetes reset-cluster` (34 s) deletes the cluster and creates a new one at once, so it doesn't free them either.

**The mode key got stripped.** I tried kubeadm mode by adding `"KubernetesMode": "kubeadm"` to `settings-store.json`:

In [ ]:
sed -i '/"KubernetesEnabled"/a\  "KubernetesMode": "kubeadm",' "$APPDATA/Docker/settings-store.json"
grep -n Kubernetes "$APPDATA/Docker/settings-store.json"    # before restart: the key is there
docker desktop restart
grep -n Kubernetes "$APPDATA/Docker/settings-store.json"    # after: gone
docker desktop kubernetes status                            # Mode: kind

Docker Desktop removed the key on restart and stayed in kind mode. (The key name is from its binaries: `grep -aoE KubernetesMode com.docker.backend.exe` finds 47 references.) My first `sed`, inserting a line with `\n`, didn't land at all; the second, with sed's append command, got stripped.

**Finding.** Turning Docker Desktop's Kubernetes off doesn't stop the cluster's containers, and they're hidden from every normal Docker command. Measure "Docker Desktop with Kubernetes off" after having had it on, and you're measuring a running cluster.

**Correction (step 08b, same day).** The mode can be set from the file. Docker Desktop writes its in-memory settings back when it stops, so my edit made while it was running got overwritten. With `docker desktop stop`, edit, `docker desktop start`, the key stays and the status says `Mode: kubeadm`.

### Clean up

In [ ]:
docker desktop kubernetes reset-cluster
docker desktop stop
sed -i 's/"KubernetesEnabled": *true/"KubernetesEnabled": false/' "$APPDATA/Docker/settings-store.json"
docker desktop start
docker rm -f 7e063ae49e37 a1cce1bb1da6 ce43b719c26d
docker rmi kindest/node:v1.35.5 docker/desktop-cloud-provider-kind:v0.6.0 docker/desktop-containerd-registry-mirror:v0.0.4
docker rmi cm-wsl-bundle:621.01
docker volume rm 12b6de8c… d152260d…      # anonymous volumes from the reset-cluster node, created 20:02 and 20:03Z
docker network rm kind
mv ~/.kube/config data/cluster-matrix/2026-10-29/08-docker-desktop/raw/kubeconfig-docker-desktop-written.yaml; rm -rf ~/.kube/cache
diff data/cluster-matrix/2026-10-29/08-docker-desktop/raw/settings-store.before.json "$APPDATA/Docker/settings-store.json"
bash bench/cluster-matrix/tools/slate-check.sh > bench/cluster-matrix/slate/08-docker-desktop-after.txt
diff bench/cluster-matrix/slate/06-minikube-after.txt bench/cluster-matrix/slate/08-docker-desktop-after.txt

settings identical to before
State:              disabled
< disk used /: 8687M
> disk used /: 8715M
< 2200 MiB, 8151 MiB, 0 %
> 2280 MiB, 8151 MiB, 14 %


I got the order wrong twice. First I set the key while Docker Desktop was running, and it wrote `true` back. Then I removed the hidden containers with `rm -f` but no `-v`, leaving their two volumes behind, the same mistake as dind in step 02. After stop, edit, start: status `disabled` (not `stopped`), no containers left, slate back to baseline. The runner fleet stayed stopped through every restart.

## 08b. Docker Desktop, kubeadm mode

**Question:** in kubeadm mode kubelet runs in Docker Desktop's own VM and starts pods on its engine, which already has an `nvidia` runtime (step 02). Does that change the answer?

**Answer:** level 50 through hostPath, but Kubernetes can't select the `nvidia` RuntimeClass.

| Setup | |
| --- | --- |
| Run | 2026-10-29 |
| Driver | 621.01, CUDA 13.5 |
| Docker Desktop | 4.88.1 |
| Kubernetes | v1.36.1, kubeadm mode, node `docker-desktop`, runtime `docker://29.7.2` (cri-dockerd) |
| Device plugin (tuned) | v0.18.2 |
| Data | [`08b-dd-kubeadm/`](../../data/cluster-matrix/2026-10-29/08b-dd-kubeadm/) |

No new noise floor: VRAM is read against step 08's, taken 40 minutes earlier (3 MiB spread).

### 1. The mode key sticks if Docker Desktop is stopped first

In [ ]:
bash bench/cluster-matrix/tools/slate-check.sh > bench/cluster-matrix/slate/08b-dd-kubeadm-before.txt
diff bench/cluster-matrix/slate/08-docker-desktop-after.txt bench/cluster-matrix/slate/08b-dd-kubeadm-before.txt
docker desktop stop
sed -i 's/"KubernetesEnabled": *false,/"KubernetesEnabled": true,/' "$APPDATA/Docker/settings-store.json"
sed -i '/"KubernetesEnabled"/a\  "KubernetesMode": "kubeadm",' "$APPDATA/Docker/settings-store.json"
grep -n Kubernetes "$APPDATA/Docker/settings-store.json"
docker desktop start
grep -n Kubernetes "$APPDATA/Docker/settings-store.json"
docker desktop kubernetes status

(slate: only disk and GPU memory moved)
## with DD stopped
8:  "KubernetesEnabled": true,
9:  "KubernetesMode": "kubeadm",
## after start
8:  "KubernetesEnabled": true,
9:  "KubernetesMode": "kubeadm",
State:              stopped
Mode:               kubeadm
Version:            v1.36.1


### 2. Cold start: 69 s, and the node already has the WSL folders

In [ ]:
docker desktop kubernetes status --format json      # polled until "status": "running"
kubectl get nodes -o wide
kubectl get node -o jsonpath='{.items[0].metadata.creationTimestamp} ready-at={...Ready lastTransitionTime}'
kubectl get runtimeclass
docker image inspect <image> --format '{{.Metadata.LastTagTime}}'

status after 44s: "status": "running"            (69.2 s from `docker desktop start`)
docker-desktop   Ready   control-plane   29s   v1.36.1   192.168.65.3   ...   Docker Desktop   ...   docker://29.7.2
2026-10-29T20:24:27Z ready-at=2026-10-29T20:24:28Z
No resources found
docker/desktop-kubernetes:kubernetes-v1.36.1-...-cri-dockerd-v0.4.4-1-debian 594MB tagged=2026-10-29 20:24:16
docker/desktop-storage-provisioner:v4.0 82.1MB tagged=2026-10-29 20:24:41
docker/desktop-vpnkit-controller:v4.0 52.6MB tagged=2026-10-29 20:24:43


The images were tagged during this start, so the 69 s is cold and includes pulling about 730 MB.

In [ ]:
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/08b-dd-kubeadm/raw a-default - - none 60
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/08b-dd-kubeadm/raw a-gpu - 1 none 20
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/08b-dd-kubeadm/raw a-hostpath - - hostpath 60
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/08b-dd-kubeadm/raw a-rc nvidia - none 30

| Pod | Result |
| --- | --- |
| default | `dxg=0`, exit 127: `libnvidia-ml.so.1: cannot open shared object file` |
| `nvidia.com/gpu: 1` | `Insufficient nvidia.com/gpu` |
| hostpath | **level 50**, with no change to the cluster |
| `runtimeClassName: nvidia` | `pods "probe-a-rc" is forbidden: pod rejected: RuntimeClass "nvidia" not found` |

The probe image came straight from Docker's image store (`imagePullPolicy: Never` works here, unlike kind mode).

**Finding.** The node is Docker Desktop's VM, so `/dev/dxg`, `/usr/lib/wsl/lib` and `/usr/lib/wsl/drivers` are already there, and a hostpath pod reaches a verified GEMM on a stock cluster. No other container-based candidate does that.

### 3. The NVIDIA runtime is there, but Kubernetes can't select it

In [ ]:
printf 'apiVersion: node.k8s.io/v1\nkind: RuntimeClass\nmetadata: {name: nvidia}\nhandler: nvidia\n' | kubectl apply -f -
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/08b-dd-kubeadm/raw b-rc nvidia - none 60
kubectl get events --field-selector involvedObject.name=probe-b-rc

runtimeclass.node.k8s.io/nvidia created
FailedCreatePodSandBox   Failed to create pod sandbox: rpc error: code = Unknown desc = RuntimeHandler "nvidia" not supported


The device plugin with `runtimeClassName: nvidia` never started for the same reason; my run waited 600 s on it before I stopped it.

**Finding.** cri-dockerd, the shim between kubelet and Docker here, doesn't pass a RuntimeClass handler through. So the toolkit route that worked on k3s and minikube is closed, though everything it needs is one layer down.

### 4. Tuned: hostPath, nothing to change on the node

In [ ]:
kubectl -n kube-system delete ds nvidia-device-plugin-daemonset
kubectl apply -f - < data/cluster-matrix/2026-10-29/03-k3s/raw/nvidia-device-plugin-v0.18.2.yml
kubectl -n kube-system patch ds nvidia-device-plugin-daemonset --type=strategic --patch-file "$(cygpath -w data/cluster-matrix/2026-10-29/04-k3d/raw/plugin-hostpath-patch.yaml)"
kubectl get node docker-desktop -o jsonpath='{.status.allocatable.nvidia\.com/gpu}'
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/08b-dd-kubeadm/raw c-gpu-hostpath - 1 hostpath 60
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/08b-dd-kubeadm/raw c-gpu-nomounts - 1 none 30
kubectl run holder ... (sleep 600, nvidia.com/gpu: 1); kubectl wait --for=condition=Ready pod/holder
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/08b-dd-kubeadm/raw c-second - 1 hostpath 20

allocatable='1' after 16s                  (22.6 s from apply)
I1029 20:36:57.142702  1 server.go:148] Registered device plugin for 'nvidia.com/gpu' with Kubelet
c-gpu-hostpath   level=50   dxg=1 libcuda_file=1 drivers_dir=1 gpu_mounts=3
c-gpu-nomounts   level=none dxg=0 ...   libnvidia-ml.so.1: cannot open shared object file
c-second         0/1 nodes are available: 1 Insufficient nvidia.com/gpu.


**Finding.** The same patched plugin and hostpath pods as k3d and kind, minus the `-v` or `extraMounts`. Rationed and working, with the same costs: GPU pods are privileged, and a pod that forgets the mounts gets scheduled with nothing.

### 5. Resources

In [ ]:
wsl -d Ubuntu -- sleep 120 &
wsl -d Ubuntu -- bash $T/bench/cluster-matrix/tools/vram.sh 20

kubeadm up: vram_mib=2226 wsl_mem_mib=1788


No CUDA context: 2226 MiB with cluster and plugin up, against 2237 MiB before step 08. WSL RAM was 1788 MiB, about 700 MiB above the ~1050 MiB with no cluster, in line with the others, but Docker Desktop restarted several times here so I wouldn't lean on it. No repeated cycles for this mode.

### Clean up

In [ ]:
docker desktop stop
cp data/cluster-matrix/2026-10-29/08-docker-desktop/raw/settings-store.before.json "$APPDATA/Docker/settings-store.json"
docker desktop start
docker desktop kubernetes status
for i in <the three docker/desktop-* images>; do docker rmi $i; done      # to read the conflicts
docker inspect --format '{{.Name}} {{.State.Status}}' <ids from the conflicts>
docker rm -f -v <ids>
docker rmi <the three images>
docker images --format '{{.Repository}}:{{.Tag}}' | grep -E 'desktop-'
diff data/cluster-matrix/2026-10-29/08-docker-desktop/raw/settings-store.before.json "$APPDATA/Docker/settings-store.json"
mv ~/.kube/config data/cluster-matrix/2026-10-29/08b-dd-kubeadm/raw/kubeconfig-docker-desktop-written.yaml; rm -rf ~/.kube/cache
bash bench/cluster-matrix/tools/slate-check.sh > bench/cluster-matrix/slate/08b-dd-kubeadm-after.txt
diff bench/cluster-matrix/slate/08-docker-desktop-after.txt bench/cluster-matrix/slate/08b-dd-kubeadm-after.txt

State:              disabled
Mode:               kind
hidden containers: 2
  /k8s_storage-provisioner_storage-provisioner_kube-system_..._0 exited
  /k8s_vpnkit-controller_vpnkit-controller_kube-system_..._0 exited
no desktop-* images
settings identical to before
< disk used /: 8715M
> disk used /: 8731M
< 2280 MiB, 8151 MiB, 14 %
> 2344 MiB, 8151 MiB, 0 %


Like kind mode, kubeadm mode leaves hidden containers behind when disabled, this time exited rather than running. Restoring the settings file while Docker Desktop was stopped put it back to `disabled` and `kind`. Back to baseline.

## 09. Comparing them

**Question:** which local Kubernetes can run GPU work here, what does each need, and which do I rebuild with the device plugin?

**Answer:** k3s: fastest, lightest, unprivileged GPU pods.

Nothing new measured; it's built from steps 01 to 08b. The tables are [`matrix.csv`](../../data/cluster-matrix/matrix.csv), [`costs.csv`](../../data/cluster-matrix/costs.csv) and [`errors.csv`](../../data/cluster-matrix/errors.csv).

### The answer

Every one reaches a verified GEMM with the GPU rationed by the scheduler, and none holds VRAM. What differs is how. k3s and minikube go through the NVIDIA runtime with unprivileged pods. The rest mount `/dev/dxg`, `/usr/lib/wsl/lib` and `/usr/lib/wsl/drivers` into privileged pods by hand, and Docker Desktop's kind mode needs a copy of the driver in every pod. k3s is three times faster and the lightest. Its catch: WSL stops its distro, and the cluster with it, when no `wsl.exe` session is attached.

### What #22 and I got wrong

- **"The GPU survives zero layers of container nesting on WSL2" is false.** It survives two in plain Docker (step 02, L2md) and three in k3d and kind. What doesn't nest is `--gpus`: it mounts files, not the directories the next layer needs.
- **There are four preconditions per boundary, not three:** `/dev/dxg`, `/usr/lib/wsl/lib` on the loader path, **`/usr/lib/wsl/drivers`** (the driver store the `lib` libraries load from), and something to put them there. Without the driver store: `nvml_init: Unable to load the NVML library` (step 02, L1m).
- **minikube's NVML failure wasn't a nesting limit.** Its node never got `/usr/lib/wsl/lib`, so the toolkit passed on a partial set (`Failed to initialize NVML: Not Supported`). One `--mount-string` fixes it (step 06).
- **I had never tested kind or k3d.** Both fail stock; two mounts fix both.
- **"Running" is not evidence.** The published device plugin on k3s and minikube's own plugin both sit at `Running` with no GPU registered.

### What works

| Candidate | The node is | Stock: best level a pod reaches | Tuned route | Rationed | Unprivileged GPU pods | A pod that doesn't request the GPU can still get it |
| --- | --- | --- | --- | --- | --- | --- |
| k3s 1.36.5 | the Ubuntu distro | **50** with `runtimeClassName: nvidia` | device plugin + one-line RuntimeClass patch | yes | **yes** | yes, any pod with the RuntimeClass |
| k3d 5.9.0 | a container | none | 2 node mounts + hostPath pods + patched plugin | yes | no | no |
| kind 0.33.0 | a container | none | `extraMounts` + hostPath pods + patched plugin | yes | no | no |
| minikube 1.39.0 | a container | 10 with its documented `--gpus=all` | add `--mount-string=/usr/lib/wsl:/usr/lib/wsl` | yes | **yes** | yes, any CUDA image |
| Rancher Desktop 1.24.0 | its own distro | **50** with hostPath | hostPath pods + patched plugin | yes | no | no |
| Docker Desktop, kind mode | a hidden container | none | driver copied into every pod | yes | no | no |
| Docker Desktop, kubeadm mode | Docker Desktop's VM | **50** with hostPath | hostPath pods + patched plugin | yes | no | no |

The method check from #22 holds: kind and Docker Desktop's kind mode gave the same stock vector (`dxg=0`, no libraries, nothing injected, exit 127), and so did k3d.

Two routes, each with a catch:

- **Runtime** (k3s, minikube): the NVIDIA runtime injects everything and pods stay unprivileged, but rationing is opt-in. On k3s any pod naming the RuntimeClass gets the GPU. On minikube, where `nvidia` is the default runtime, any image setting `NVIDIA_VISIBLE_DEVICES=all` gets it, which is every `nvidia/cuda` image.
- **Mount** (k3d, kind, Rancher Desktop, Docker Desktop kubeadm): the plugin counts the card through the same three mounts, and nothing reaches the GPU without mounting them. But every GPU pod is privileged with three hostPath volumes, and one that forgets the mounts gets scheduled with nothing.

### What it costs

| Candidate | Cold to Ready (s) | Cold to level 50 (s) | WSL RAM added (MiB) | Disk | Teardown | Left behind |
| --- | --- | --- | --- | --- | --- | --- |
| k3s | **8 to 9** | **17 to 19** (unrationed; plugin +4) | **364 to 382** | 0.7 GB | 3 to 4 s | `/etc/rancher/node/password` |
| k3d | 29 | 54 to 64 | 641 to 667 | 0.5 GB images | 2 s | nothing |
| kind | 46 to 56 | 71 to 80 | 599 to 628 | 1.3 GB image | 1.3 s | the `kind` network |
| minikube | 86 to 95 | 107 to 116 | 617 to 633 | 0.5 GB + preload | 13 s | kicbase images |
| Rancher Desktop | 43 | 66 (on 1.25) | about 590 | **6.9 GB** | reset + 3 s | nothing, if reset first |
| Docker Desktop kind | 82 | 81 (warm) | couldn't measure cleanly | 2.2 GB | by hand | running hidden containers |
| Docker Desktop kubeadm | 69 | not cycled | about 700 | 0.7 GB | by hand | exited hidden containers |

VRAM: no candidate held a CUDA context; every reading stayed inside that session's floor and drift. The floor varied by session from 1 to 39 MiB (`01: 1, 02: 10, 03: 3, 04: 39, 05: 2, 06: 2, 07: 2, 08: 3`) and the card idled between 2190 and 2336 MiB across the day. A context is 113 MiB (step 01), so a yes couldn't be missed.

Repeatability: the plain Docker ladder gave the same answer on all ten rungs in three runs. k3s, k3d, kind and minikube each had three timed cycles that agreed. Rancher Desktop and Docker Desktop got one each, Docker Desktop's warm. Nothing was flaky.

### Recommendation: k3s with the device plugin

The harnesses here need a working GPU, a scheduler that can say nobody else holds it, and nothing that taxes the measurement.

- **k3s**: fastest from nothing (17 to 19 s), lightest (about 370 MiB), unprivileged pods, and one manifest plus a one-line patch to ration the GPU. Two open items, neither on the GPU path:
  1. **Keeping the distro alive.** A held `wsl.exe` session works (03); a durable way to hold one (a logon task or a WSL setting) is untested. It decides whether k3s is a server or only a session.
  2. **Opt-in rationing.** A pod naming the RuntimeClass without requesting the GPU still gets it, and #10's exclusivity check, which reads `nvidia.com/gpu` allocations, can't see it. Either an admission policy makes every GPU workload request the resource, or the toolkit ignores the variable from unprivileged pods and the plugin switches to a volume-mounts device list. Untested.
- **minikube** is runner-up on the same route, but six times slower to start, with a wider rationing hole.
- **On the mount route**, Rancher Desktop and Docker Desktop kubeadm need the least setup (their nodes already have the folders) but cost privileged pods, and Rancher Desktop defaults to an unsupported Kubernetes.

## 10. The rebuild: k3s as gpu-lab

**Decision:** k3s in the Ubuntu distro with the NVIDIA device plugin, kept alive by a logon task, with an admission policy that sends every GPU pod through the scheduler.

The reasons are in step 09. Its two weak spots, WSL stopping the distro and opt-out rationing, are fixed below.

| Setup | |
| --- | --- |
| Run | 2026-10-29 |
| Driver | 621.01, CUDA 13.5 |
| k3s | v1.36.5+k3s1 |
| Toolkit | nvidia-container-toolkit 1.20.1-1, in Ubuntu |
| Device plugin | v0.18.2 |
| Policy | `ValidatingAdmissionPolicy`, `admissionregistration.k8s.io/v1` |
| Data | [`10-rebuild/`](../../data/cluster-matrix/2026-10-29/10-rebuild/) |

This closes #22's last item: k3s rebuilt with the device plugin and `nvidia.com/gpu` allocatable. It also gives #10 something to read: with the policy, a pod can only touch the GPU by requesting it, so "is `nvidia.com/gpu` allocated" answers "is anyone else on the card".

### 1. Install: Ready in 9 s, kubeconfig readable

In [ ]:
bash bench/cluster-matrix/tools/slate-check.sh > bench/cluster-matrix/slate/10-rebuild-before.txt
diff bench/cluster-matrix/slate/07-rancher-desktop-after.txt bench/cluster-matrix/slate/10-rebuild-before.txt
# bench/cluster-matrix/tools/k3s-rebuild.sh, as root in Ubuntu:
curl -sfL https://get.k3s.io | K3S_NODE_NAME=gpu-lab K3S_KUBECONFIG_MODE=644 sh -

(slate: only GPU memory moved)
install to Ready: 9.0 s
k3s version v1.36.5+k3s1 (3dd98cc5)
-rw-r--r-- 1 root root 2945 Oct 29 17:21 /etc/rancher/k3s/k3s.yaml


`K3S_KUBECONFIG_MODE=644` is new since step 03: k3s rewrites the file root-only on every start ([gpu-debug.md](../gpu-debug.md)), which breaks `kubectl` inside WSL after each restart.

### 2. Plugin and policy as auto-deploy manifests

Both go in `/var/lib/rancher/k3s/server/manifests/`, which k3s applies on every start, so they come back with the cluster.

In [ ]:
# bench/cluster-matrix/tools/k3s-manifests.sh, as root in Ubuntu:
k3s kubectl get runtimeclass nvidia
k3s kubectl patch --local -f data/cluster-matrix/2026-10-29/03-k3s/raw/nvidia-device-plugin-v0.18.2.yml --type=json \
  -p '[{"op":"add","path":"/spec/template/spec/runtimeClassName","value":"nvidia"}]' -o yaml \
  > /var/lib/rancher/k3s/server/manifests/nvidia-device-plugin.yaml
cp bench/cluster-matrix/gpu-admission-policy.yaml /var/lib/rancher/k3s/server/manifests/
k3s kubectl get node gpu-lab -o jsonpath='{.status.allocatable.nvidia\.com/gpu}'
k3s kubectl get validatingadmissionpolicy,validatingadmissionpolicybinding
k3s kubectl get addons -n kube-system | grep -E 'nvidia|gpu'

runtimeclass: nvidia -> nvidia
28:      runtimeClassName: nvidia
allocatable=1 after 9.6 s
validatingadmissionpolicy.../gpu-request-and-runtimeclass-together   2   <unset>   9s
validatingadmissionpolicybinding.../gpu-request-and-runtimeclass-together   gpu-request-and-runtimeclass-together
gpu-admission-policy   /var/lib/rancher/k3s/server/manifests/gpu-admission-policy.yaml   4d974e66...
nvidia-device-plugin   /var/lib/rancher/k3s/server/manifests/nvidia-device-plugin.yaml   a8314b38...


The RuntimeClass appeared a few seconds after Ready; my first check, right at Ready, read `runtimeclasses.node.k8s.io "nvidia" not found`.

The policy (`bench/cluster-matrix/gpu-admission-policy.yaml`) has two rules for every pod outside `kube-system`: `runtimeClassName: nvidia` requires a `nvidia.com/gpu` limit, and a `nvidia.com/gpu` limit requires `runtimeClassName: nvidia`. `kube-system` is exempt because the device plugin runs with the RuntimeClass and requests nothing.

### 3. kubectl on Windows

In [ ]:
wsl -d Ubuntu -u root -- cat /etc/rancher/k3s/k3s.yaml \
  | sed 's/127.0.0.1/localhost/; s/: default$/: gpu-lab/; s/name: default$/name: gpu-lab/; s/current-context: default/current-context: gpu-lab/' \
  > ~/.kube/config
kubectl config get-contexts

*         gpu-lab   gpu-lab   gpu-lab


### 4. The cluster died within minutes, before the keepalive

Halfway through the policy tests, `kubectl` on Windows lost the cluster:

```text
error: failed to create configmap: Post "https://localhost:6443/api/v1/namespaces/default/configmaps?...": dial tcp [::1]:6443: connectex: No connection could be made because the target machine actively refused it.
gpu-lab   Ready   control-plane   62s   v1.36.5+k3s1          (when it came back: a new start)
```

Step 03's finding, live: my last `wsl.exe` call ended, WSL stopped the distro, and k3s went with it. `kubectl` from Windows reaches `localhost` without a WSL session, so it keeps nothing alive.

### 5. The keepalive: a hidden logon task that has to loop

First version: a logon task running `conhost.exe --headless wsl.exe -d Ubuntu --exec /bin/sleep infinity`, restart on failure every minute, no time limit, no window, no admin.

```powershell
$a = New-ScheduledTaskAction -Execute "$env:WINDIR\System32\conhost.exe" -Argument '--headless wsl.exe -d Ubuntu --exec /bin/sleep infinity'
$t = New-ScheduledTaskTrigger -AtLogOn -User "$env:USERDOMAIN\$env:USERNAME"
$s = New-ScheduledTaskSettingsSet -ExecutionTimeLimit ([TimeSpan]::Zero) -RestartCount 999 -RestartInterval (New-TimeSpan -Minutes 1) -AllowStartIfOnBatteries -DontStopIfGoingOnBatteries -StartWhenAvailable -MultipleInstances IgnoreNew
$p = New-ScheduledTaskPrincipal -UserId "$env:USERDOMAIN\$env:USERNAME" -LogonType Interactive -RunLevel Limited
Register-ScheduledTask -TaskName 'gpu-lab keepalive' -Action $a -Trigger $t -Settings $s -Principal $p
Start-ScheduledTask -TaskName 'gpu-lab keepalive'
```

I left it 150 s with no `wsl.exe` calls of my own, only `kubectl` from Windows:

```text
start: systemd 1534549 node_created=2026-10-29T21:21:29Z restarts_total=10 21:29:49
after 150 s (kubectl only): gpu-lab   Ready   control-plane   10m   v1.36.5+k3s1 restarts_total=10 21:32:19
end:   systemd 1534549
```

Same systemd start, no pod restarts (the 10 are from finding 4).

Then I killed the distro, standing in for a crash or a WSL update:

In [ ]:
wsl --terminate Ubuntu
# poll kubectl get nodes for up to 180 s
Get-ScheduledTask -TaskName 'gpu-lab keepalive' | Get-ScheduledTaskInfo

kubectl sees Ready again after 180s       (it did not; the loop timed out)
LastRunTime        : 29/10/2026 17:29:35
LastTaskResult     : 0
Unable to connect to the server: dial tcp [::1]:6443: connectex: No connection could be made ...


**Finding.** On terminate, `wsl.exe` exits cleanly, Task Scheduler sees success, and restart on failure never fires. The cluster stayed down. The action has to loop itself:

```powershell
$loop = 'while ($true) { & wsl.exe -d Ubuntu --exec /bin/sleep infinity; Start-Sleep -Seconds 5 }'
$a = New-ScheduledTaskAction -Execute "$env:WINDIR\System32\conhost.exe" -Argument "--headless powershell.exe -NoProfile -NonInteractive -Command `"$loop`""
Set-ScheduledTask -TaskName 'gpu-lab keepalive' -Action $a
Start-ScheduledTask -TaskName 'gpu-lab keepalive'
```

Same test again:

```text
cluster up: gpu-lab   Ready   control-plane   16m   v1.36.5+k3s1
terminate at 21:37:50
kubectl sees Ready again after 9s at 21:38:02
nvidia.com/gpu allocatable=1 (20s after terminate)
t6-after-terminate   level=50   dxg=1 libcuda_file=0 drivers_dir=1 gpu_mounts=143
```

**Finding.** With the loop the cluster recovers on its own: Ready 9 s after the kill, GPU schedulable at 20 s, a verified GEMM straight after. Untested: a reboot and fresh logon, which is what the logon trigger is for.

### 6. The policy: every way to the GPU goes through the scheduler

In [ ]:
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/10-rebuild/raw t1-rc-gpu nvidia 1 none 60
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/10-rebuild/raw t2-rc-only nvidia - none 20
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/10-rebuild/raw t3-gpu-only - 1 none 20
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/10-rebuild/raw t4-neither - - none 60
kubectl run holder ... (runtimeClassName nvidia, nvidia.com/gpu: 1, sleep 600); kubectl wait --for=condition=Ready pod/holder
bash bench/cluster-matrix/tools/kpod.sh data/cluster-matrix/2026-10-29/10-rebuild/raw t5-second nvidia 1 none 20

| Pod | Result |
| --- | --- |
| RuntimeClass + `nvidia.com/gpu: 1` | **level 50**, unprivileged, 143 injected mounts |
| RuntimeClass only | **denied**: `ValidatingAdmissionPolicy 'gpu-request-and-runtimeclass-together' ... denied request: runtimeClassName nvidia without a nvidia.com/gpu limit would use the GPU without the scheduler knowing. Add resources.limits nvidia.com/gpu: 1.` |
| `nvidia.com/gpu: 1` only | **denied**: `... nvidia.com/gpu requested without runtimeClassName nvidia would hold the GPU with no driver. Add runtimeClassName: nvidia.` |
| neither | runs with no GPU (exit 127), as it should |
| a second request while `holder` has the GPU | `0/1 nodes are available: 1 Insufficient nvidia.com/gpu.` |

**Finding.** Both traps from step 03 are rejected at admission, with a message saying what to add. On the default runtime (runc), a pod that asks for nothing gets nothing, even from a CUDA image. Out of scope, as in the runner fleet post: init containers that request the GPU, `kube-system`, and a privileged pod that mounts the WSL paths by hand. Those are deliberate acts.

### 7. Idle cost

In [ ]:
wsl -d Ubuntu -- bash $T/bench/cluster-matrix/tools/vram.sh 20
kubectl get pods -A

idle gpu-lab: vram_mib=2190 wsl_mem_mib=1293
kube-system: coredns, local-path-provisioner, metrics-server, nvidia-device-plugin, svclb-traefik, traefik (Running); helm-install-traefik x2 (Completed)


No VRAM held, and about 240 MiB of WSL RAM above an empty machine.

### The new standing state

`bench/cluster-matrix/slate/10-rebuild-after.txt` replaces the blank slate as the reference. Its diff from 07's slate is exactly the cluster (k3s binaries and units, `/etc/rancher`, `/var/lib/rancher`, `/var/lib/kubelet`, 24 mounts, 8 CNI interfaces, 245 iptables rules, about 1.8 GB of disk, a `gpu-lab` kubeconfig on Windows), plus the `gpu-lab keepalive` task.